In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch

print(torch.cuda.is_available())

In [ ]:
import os

print(os.listdir("/kaggle/input/datasets/"))

In [ ]:
os.listdir("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")

In [ ]:
os.listdir("/kaggle/input/datasets/hiro002/dermacon-in-dataset")

In [ ]:
os.listdir("/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629")

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
ham_path="/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"

ham=pd.read_csv(ham_path)

print(ham.head())

In [ ]:
for root,dirs,files in os.walk("/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root,file))

In [ ]:
isic=pd.read_csv(
"/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/metadata.csv"
)

print(isic.head())

In [ ]:
# =======================================
# REBUILD ISIC IMAGE PATHS
# =======================================

import os
import glob

isic_image_dir = "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/images"

# Find all image files
isic_image_files = glob.glob(
    os.path.join(isic_image_dir, "**", "*"),
    recursive=True
)

# Keep image files only
isic_image_files = [
    p for p in isic_image_files
    if p.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
]

# Create filename -> full path mapping
isic_image_map = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in isic_image_files
}

# Match image path using isic_id
isic["image_path"] = isic["isic_id"].astype(str).map(isic_image_map)

print("=======================================")
print("ISIC IMAGE PATH MATCHING")
print("=======================================")

print("Total ISIC records:", len(isic))
print("Valid image paths:", isic["image_path"].notna().sum())
print("Missing image paths:", isic["image_path"].isna().sum())

print("\nExample paths:")
print(isic[["isic_id", "image_path"]].head())

In [ ]:
# =======================================
# REBUILD 4-CLASS DATASET
# =======================================

import pandas as pd

# ---------------------------------------
# ISIC: Keep Melanoma + BCC
# ---------------------------------------

isic_clean = isic[
    isic["class_name"].isin([
        "Melanoma",
        "Basal Cell Carcinoma"
    ])
].copy()

isic_clean["label"] = isic_clean["class_name"].map({
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1
})

isic_clean["dataset"] = "ISIC"


# ---------------------------------------
# NEW SCC
# ---------------------------------------

scc_clean = new_scc_df.copy()

scc_clean["label"] = 2
scc_clean["dataset"] = "NEW_SCC"


# ---------------------------------------
# NEW MCC
# ---------------------------------------

mcc_clean = new_mcc_df.copy()

mcc_clean["label"] = 3
mcc_clean["dataset"] = "NEW_MCC"


# ---------------------------------------
# Combine all 4 classes
# ---------------------------------------

combined_df = pd.concat(
    [
        isic_clean[["image_name", "image_path", "class_name", "label", "dataset"]],
        scc_clean[["image_name", "image_path", "class_name", "label", "dataset"]],
        mcc_clean[["image_name", "image_path", "class_name", "label", "dataset"]]
    ],
    ignore_index=True
)

# ---------------------------------------
# Remove invalid paths
# ---------------------------------------

combined_df = combined_df[
    combined_df["image_path"].notna()
].copy()

# Remove duplicate image paths
combined_df = combined_df.drop_duplicates(
    subset=["image_path"]
).reset_index(drop=True)


print("=======================================")
print("4-CLASS DATASET CREATED")
print("=======================================")

print("Total images:", len(combined_df))

print("\nClass distribution:")
print(combined_df["class_name"].value_counts())

print("\nLabel distribution:")
print(combined_df["label"].value_counts().sort_index())

print("\nDataset distribution:")
print(combined_df["dataset"].value_counts())

In [ ]:
# =======================================
# ISIC CLASS LABEL CHECK
# =======================================

print("=======================================")
print("ISIC METADATA COLUMNS")
print("=======================================")

print(isic.columns.tolist())

print("\n=======================================")
print("BENIGN / MALIGNANT DISTRIBUTION")
print("=======================================")

print(isic["benign_malignant"].value_counts(dropna=False))

print("\n=======================================")
print("MELANOMA TYPE VALUES")
print("=======================================")

print(isic["mel_type"].value_counts(dropna=False))

In [ ]:
# =======================================
# CHECK DIAGNOSIS / MELANOMA CLASS LABELS
# =======================================

print("=======================================")
print("DIAGNOSIS DISTRIBUTION")
print("=======================================")

print(
    isic["diagnosis"].value_counts(
        dropna=False
    ).head(30)
)


print("\n=======================================")
print("MEL_CLASS DISTRIBUTION")
print("=======================================")

print(
    isic["mel_class"].value_counts(
        dropna=False
    )
)

In [ ]:
# =======================================
# CREATE 4-CLASS LABELS FROM ISIC DIAGNOSIS
# =======================================

print("\n")
print("=======================================")
print("CREATING 4-CLASS LABELS")
print("=======================================")

def map_cancer_class(diagnosis):

    if pd.isna(diagnosis):
        return np.nan

    diagnosis = str(diagnosis).strip().lower()

    if diagnosis == "melanoma":
        return "Melanoma"

    elif diagnosis == "basal cell carcinoma":
        return "Basal Cell Carcinoma"

    elif diagnosis == "squamous cell carcinoma":
        return "Squamous Cell Carcinoma"

    elif "merkel" in diagnosis:
        return "Merkel Cell Carcinoma"

    else:
        return np.nan


# Create class_name column
isic["class_name"] = isic["diagnosis"].apply(
    map_cancer_class
)


# =======================================
# KEEP ONLY 4 TARGET CLASSES
# =======================================

target_classes = [
    "Melanoma",
    "Basal Cell Carcinoma",
    "Squamous Cell Carcinoma",
    "Merkel Cell Carcinoma"
]

isic_clean = isic[
    isic["class_name"].isin(target_classes)
].copy()


# =======================================
# DISPLAY CLASS DISTRIBUTION
# =======================================

print("\n")
print("=======================================")
print("4-CLASS DISTRIBUTION")
print("=======================================")

print(
    isic_clean["class_name"].value_counts()
)


print("\n")
print("Total 4-class images:",
      len(isic_clean))


print("\n")
print("=======================================")
print("4-CLASS LABEL CREATION COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# CHECK FOR MERKEL CELL CARCINOMA
# =======================================

print("=======================================")
print("CHECKING FOR MERKEL CELL CARCINOMA")
print("=======================================")

merkel_rows = isic[
    isic["diagnosis"]
    .astype(str)
    .str.contains("merkel", case=False, na=False)
]

print("Merkel-related records:")
print(len(merkel_rows))

if len(merkel_rows) > 0:
    print("\nMerkel diagnosis values:")
    print(
        merkel_rows["diagnosis"].value_counts()
    )

else:
    print(
        "\nNo Merkel Cell Carcinoma records "
        "found in this metadata."
    )

In [ ]:
# =======================================
# CHECK UPLOADED MCC DATASET
# =======================================

import os

print("=======================================")
print("KAGGLE INPUT DATASETS")
print("=======================================")

input_path = "/kaggle/input"

datasets = os.listdir(input_path)

for i, name in enumerate(datasets):
    print(i, "=", name)

print("\n=======================================")
print("DATASET CHECK COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# CHECK INSIDE DATASETS FOLDER
# =======================================

import os

datasets_path = "/kaggle/input/datasets"

print("=======================================")
print("DATASETS FOLDER CONTENT")
print("=======================================")

for name in os.listdir(datasets_path):
    print(name)

print("\n=======================================")
print("CHECK COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# FIND MCC DATASET / FOLDERS
# =======================================

import os

base_path = "/kaggle/input/datasets"

print("=======================================")
print("SEARCHING FOR MCC DATASET")
print("=======================================")

for user_folder in os.listdir(base_path):

    user_path = os.path.join(
        base_path,
        user_folder
    )

    if os.path.isdir(user_path):

        print(f"\n[{user_folder}]")

        for root, dirs, files in os.walk(user_path):

            # Show folders/files containing MCC or Merkel
            matches = [
                x for x in dirs + files
                if "mcc" in x.lower()
                or "merkel" in x.lower()
            ]

            for match in matches:
                print(
                    os.path.join(root, match)
                )

print("\n=======================================")
print("MCC SEARCH COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# MCC DATASET STRUCTURE CHECK
# =======================================

import os

mcc_paths = [
    "/kaggle/input/datasets/quantumcoders05/mcc-dataset",
    "/kaggle/input/datasets/quantumcoders05/mccdataset"
]

print("=======================================")
print("MCC DATASET CHECK")
print("=======================================")

for path in mcc_paths:

    print("\nDATASET:")
    print(path)

    if not os.path.exists(path):
        print("NOT FOUND")
        continue

    image_files = []

    for root, dirs, files in os.walk(path):

        for file in files:

            if file.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp", ".webp")
            ):
                image_files.append(
                    os.path.join(root, file)
                )

    print("Image count:", len(image_files))

    print("\nFirst 10 images:")

    for image_path in image_files[:10]:
        print(image_path)

print("\n=======================================")
print("MCC DATASET CHECK COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# MCC IMAGE FILE CHECK
# =======================================

import os
from PIL import Image

mcc_roots = [
    "/kaggle/input/datasets/quantumcoders05/mcc-dataset",
    "/kaggle/input/datasets/quantumcoders05/mccdataset"
]

mcc_images = []

for root in mcc_roots:

    for current_root, dirs, files in os.walk(root):

        for file in files:

            if file.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp", ".webp")
            ):

                path = os.path.join(
                    current_root,
                    file
                )

                try:
                    with Image.open(path) as img:

                        width, height = img.size

                        if width > 0 and height > 0:
                            mcc_images.append({
                                "path": path,
                                "filename": file,
                                "width": width,
                                "height": height
                            })

                except Exception as e:

                    print("Invalid image:", path)
                    print("Error:", e)


# Remove exact duplicate paths
unique_paths = {}

for item in mcc_images:
    unique_paths[item["path"]] = item

mcc_images = list(unique_paths.values())


print("=======================================")
print("MCC IMAGE VALIDATION")
print("=======================================")

print("Valid MCC images:", len(mcc_images))

print("\nImages:")

for i, item in enumerate(mcc_images, 1):

    print(
        f"{i}. {item['filename']} "
        f"({item['width']}x{item['height']})"
    )

print("\n=======================================")
print("MCC IMAGE VALIDATION COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# COMBINE ISIC 3 CLASSES + MCC
# =======================================

print("=======================================")
print("CREATING 4-CLASS DATASET")
print("=======================================")

# ---------------------------------------
# ISIC 3-class data
# ---------------------------------------

isic_3class = isic_clean[
    isic_clean["class_name"].isin([
        "Melanoma",
        "Basal Cell Carcinoma",
        "Squamous Cell Carcinoma"
    ])
].copy()


# ---------------------------------------
# Create MCC dataframe
# ---------------------------------------

mcc_df = pd.DataFrame(mcc_images)

mcc_df["class_name"] = "Merkel Cell Carcinoma"

mcc_df = mcc_df[
    ["path", "filename", "width", "height", "class_name"]
].copy()

mcc_df = mcc_df.rename(
    columns={"path": "image_path"}
)


# ---------------------------------------
# Keep required ISIC columns
# ---------------------------------------

isic_for_training = isic_3class[
    ["image_path", "class_name"]
].copy()


# ---------------------------------------
# Combine
# ---------------------------------------

combined_dataset = pd.concat(
    [
        isic_for_training,
        mcc_df[
            ["image_path", "class_name"]
        ]
    ],
    ignore_index=True
)


# ---------------------------------------
# Shuffle
# ---------------------------------------

combined_dataset = combined_dataset.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# ---------------------------------------
# Class mapping
# ---------------------------------------

class_names = [
    "Melanoma",
    "Basal Cell Carcinoma",
    "Squamous Cell Carcinoma",
    "Merkel Cell Carcinoma"
]

class_to_idx = {
    name: idx
    for idx, name in enumerate(class_names)
}


combined_dataset["label"] = (
    combined_dataset["class_name"]
    .map(class_to_idx)
)


# =======================================
# DISPLAY RESULTS
# =======================================

print("\n=======================================")
print("4-CLASS DISTRIBUTION")
print("=======================================")

print(
    combined_dataset["class_name"]
    .value_counts()
)


print("\nTotal images:",
      len(combined_dataset))


print("\nClass mapping:")

for idx, name in enumerate(class_names):
    print(idx, "=", name)


print("\n=======================================")
print("4-CLASS DATASET READY")
print("=======================================")

In [ ]:
# =======================================
# 4-CLASS TRAIN / VALIDATION SPLIT
# =======================================

from sklearn.model_selection import train_test_split

print("=======================================")
print("CREATING TRAIN / VALIDATION SPLIT")
print("=======================================")

# ---------------------------------------
# Stratified split
# ---------------------------------------

train_df, val_df = train_test_split(
    combined_dataset,
    test_size=0.20,
    random_state=42,
    stratify=combined_dataset["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)


# =======================================
# DISPLAY RESULTS
# =======================================

print("\nTrain images:", len(train_df))
print("Validation images:", len(val_df))

print("\n=======================================")
print("TRAIN CLASS DISTRIBUTION")
print("=======================================")

print(
    train_df["class_name"].value_counts()
)


print("\n=======================================")
print("VALIDATION CLASS DISTRIBUTION")
print("=======================================")

print(
    val_df["class_name"].value_counts()
)


print("\n=======================================")
print("TRAIN / VALIDATION SPLIT READY")
print("=======================================")

In [ ]:
# =======================================
# 4-CLASS PYTORCH DATASET + DATALOADER
# =======================================

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

print("=======================================")
print("CREATING PYTORCH DATASETS")
print("=======================================")


# =======================================
# DATASET CLASS
# =======================================

class SkinCancerDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["image_path"]
        label = int(row["label"])

        image = Image.open(
            image_path
        ).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label


# =======================================
# CREATE DATASETS
# =======================================

train_dataset = SkinCancerDataset(
    train_df,
    transform=train_transform
)

val_dataset = SkinCancerDataset(
    val_df,
    transform=test_transform
)


# =======================================
# DATALOADERS
# =======================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# =======================================
# CHECK
# =======================================

print("\nTrain dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

print("\nTrain batches:", len(train_loader))
print("Validation batches:", len(val_loader))

print("\n=======================================")
print("PYTORCH DATASETS READY")
print("=======================================")

In [ ]:
# =======================================
# TEST IMAGE BATCH LOADING
# =======================================

print("=======================================")
print("TESTING IMAGE BATCH")
print("=======================================")

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

print("\nLabels in first batch:")
print(labels.tolist())

print("\nImage value range:")
print("Min:", images.min().item())
print("Max:", images.max().item())

print("\n=======================================")
print("IMAGE BATCH LOADING SUCCESSFUL")
print("=======================================")

In [ ]:
# =======================================
# TEST MODEL FORWARD PASS
# =======================================

print("=======================================")
print("TESTING MODEL FORWARD PASS")
print("=======================================")

model.eval()

with torch.no_grad():
    test_images = images.to(device)
    outputs = model(test_images)

print("Input shape :", test_images.shape)
print("Output shape:", outputs.shape)

print("\nModel output shape should be:")
print("(32, 4)")

print("\n=======================================")
print("MODEL FORWARD PASS COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# SWITCH TO CPU
# =======================================

device = torch.device("cpu")

model = model.to(device)

print("=======================================")
print("CPU DEVICE READY")
print("=======================================")
print("Device:", device)

In [ ]:
# =======================================
# TEST MODEL FORWARD PASS ON CPU
# =======================================

print("=======================================")
print("TESTING MODEL FORWARD PASS - CPU")
print("=======================================")

model.eval()

with torch.no_grad():
    test_images = images.to(device)
    outputs = model(test_images)

print("Input shape :", test_images.shape)
print("Output shape:", outputs.shape)

print("\n=======================================")
print("MODEL FORWARD PASS SUCCESSFUL")
print("=======================================")

In [ ]:
# =======================================
# FIND MCC DATASET
# =======================================

import os

root = "/kaggle/input/datasets"

print("=======================================")
print("SEARCHING FOR MCC DATASET")
print("=======================================")

for current_root, dirs, files in os.walk(root):

    # Keep output manageable
    if "mcc" in current_root.lower() or "merkel" in current_root.lower():
        print("\nFOLDER:")
        print(current_root)

        if files:
            print("Files:", files[:20])

print("\n=======================================")
print("MCC SEARCH COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# MCC DATASET IMAGE COUNT
# =======================================

import os

mcc_roots = [
    "/kaggle/input/datasets/quantumcoders05/mcc-dataset",
    "/kaggle/input/datasets/quantumcoders05/mccdataset"
]

image_extensions = (
    ".jpg", ".jpeg", ".png", ".webp", ".bmp"
)

print("=======================================")
print("MCC DATASET IMAGE COUNTS")
print("=======================================")

for root in mcc_roots:

    count = 0
    examples = []

    for current_root, dirs, files in os.walk(root):

        for file in files:

            if file.lower().endswith(image_extensions):

                count += 1

                if len(examples) < 10:
                    examples.append(
                        os.path.join(current_root, file)
                    )

    print("\nDataset:")
    print(root)

    print("Total images:", count)

    print("\nExample images:")

    for image in examples:
        print(image)

print("\n=======================================")
print("MCC IMAGE COUNT CHECK COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# MCC IMAGE VISUAL INSPECTION
# =======================================

from PIL import Image
import os

mcc_roots = [
    "/kaggle/input/datasets/quantumcoders05/mcc-dataset",
    "/kaggle/input/datasets/quantumcoders05/mccdataset"
]

image_extensions = (
    ".jpg", ".jpeg", ".png", ".webp", ".bmp"
)

mcc_images = []

for root in mcc_roots:

    for current_root, dirs, files in os.walk(root):

        for file in files:

            if file.lower().endswith(image_extensions):

                mcc_images.append(
                    os.path.join(current_root, file)
                )

print("=======================================")
print("MCC IMAGES FOUND")
print("=======================================")
print("Total:", len(mcc_images))

for i, path in enumerate(mcc_images):
    print(i, ":", os.path.basename(path))

print("\n=======================================")
print("MCC IMAGE LIST READY")
print("=======================================")

In [ ]:
# =======================================
# VISUAL INSPECTION OF MCC IMAGES
# =======================================

import matplotlib.pyplot as plt
from PIL import Image
import math

cols = 4
rows = math.ceil(len(mcc_images) / cols)

plt.figure(figsize=(16, rows * 4))

for i, image_path in enumerate(mcc_images):

    try:
        img = Image.open(image_path).convert("RGB")

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(
            f"{i}: {os.path.basename(image_path)[:25]}"
        )
        plt.axis("off")

    except Exception as e:
        print("Error:", image_path, e)

plt.tight_layout()
plt.show()

In [ ]:
# =======================================
# RECREATE MCC IMAGE LIST
# =======================================

import os

mcc_roots = [
    "/kaggle/input/datasets/quantumcoders05/mcc-dataset",
    "/kaggle/input/datasets/quantumcoders05/mccdataset"
]

image_extensions = (
    ".jpg", ".jpeg", ".png", ".webp", ".bmp"
)

mcc_images = []

for root in mcc_roots:

    for current_root, dirs, files in os.walk(root):

        for file in files:

            if file.lower().endswith(image_extensions):

                mcc_images.append(
                    os.path.join(current_root, file)
                )

print("=======================================")
print("MCC IMAGE LIST READY")
print("=======================================")
print("Total MCC images:", len(mcc_images))

In [ ]:
# =======================================
# VISUAL INSPECTION OF MCC IMAGES
# =======================================

import matplotlib.pyplot as plt
from PIL import Image
import math
import os

cols = 4
rows = math.ceil(len(mcc_images) / cols)

plt.figure(figsize=(16, rows * 4))

for i, image_path in enumerate(mcc_images):

    try:
        img = Image.open(image_path).convert("RGB")

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(
            f"{i}: {os.path.basename(image_path)[:25]}"
        )
        plt.axis("off")

    except Exception as e:
        print("Error:", image_path, e)

plt.tight_layout()
plt.show()

In [ ]:
# =======================================
# MCC IMAGE REVIEW - PAGED
# =======================================

import matplotlib.pyplot as plt
from PIL import Image
import math
import os

print("=======================================")
print("MCC IMAGE REVIEW")
print("=======================================")
print("Total images:", len(mcc_images))

images_per_page = 6
total_pages = math.ceil(len(mcc_images) / images_per_page)

for page in range(total_pages):

    start = page * images_per_page
    end = min(start + images_per_page, len(mcc_images))

    current_images = mcc_images[start:end]

    fig, axes = plt.subplots(
        2, 3,
        figsize=(15, 9)
    )

    axes = axes.flatten()

    for j, image_path in enumerate(current_images):

        try:

            img = Image.open(
                image_path
            ).convert("RGB")

            axes[j].imshow(img)

            axes[j].set_title(
                f"INDEX {start + j}\n"
                f"{os.path.basename(image_path)[:35]}",
                fontsize=9
            )

            axes[j].axis("off")

        except Exception as e:

            axes[j].text(
                0.5,
                0.5,
                f"ERROR\n{e}",
                ha="center",
                va="center"
            )

            axes[j].axis("off")

    # Hide unused boxes
    for j in range(len(current_images), 6):
        axes[j].axis("off")

    plt.suptitle(
        f"MCC IMAGE REVIEW - PAGE {page + 1}/{total_pages}",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()

print("\n=======================================")
print("MCC IMAGE REVIEW COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# SELECT USABLE MCC CLINICAL IMAGES
# =======================================

usable_mcc_indices = [
    3, 5, 7, 9, 11, 12,
    14, 15, 16, 17, 18,
    19, 21, 22
]

mcc_selected = [
    mcc_images[i]
    for i in usable_mcc_indices
]

print("=======================================")
print("SELECTED MCC CLINICAL IMAGES")
print("=======================================")

print("Selected:", len(mcc_selected))

for i, path in zip(usable_mcc_indices, mcc_selected):
    print(f"{i}: {os.path.basename(path)}")

print("\n=======================================")
print("MCC SELECTION COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# CREATE MCC CLASS DATA
# =======================================

import pandas as pd
import os

print("=======================================")
print("CREATING MCC CLASS DATA")
print("=======================================")

mcc_df = pd.DataFrame({
    "image_path": mcc_selected,
    "class_name": "Merkel Cell Carcinoma",
    "label": 3
})

print("MCC images:", len(mcc_df))
print("\nClass:")
print(mcc_df["class_name"].value_counts())

print("\nExample MCC paths:")
print(mcc_df.head())

print("\n=======================================")
print("MCC CLASS DATA READY")
print("=======================================")

In [ ]:
# =======================================
# CHECK DATAFRAME VARIABLES
# =======================================

print("=======================================")
print("CHECKING DATAFRAME VARIABLES")
print("=======================================")

for name, value in list(globals().items()):

    if isinstance(value, pd.DataFrame):

        print(
            f"{name} -> "
            f"rows: {len(value)}, "
            f"columns: {value.columns.tolist()}"
        )

print("\n=======================================")
print("DATAFRAME CHECK COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# RECREATE ISIC 3-CLASS DATA
# =======================================

print("=======================================")
print("RECREATING ISIC 3-CLASS DATA")
print("=======================================")

# Keep only the 3 target diagnoses
target_diagnoses = [
    "melanoma",
    "basal cell carcinoma",
    "squamous cell carcinoma"
]

isic_3class = isic[
    isic["diagnosis"].isin(target_diagnoses)
].copy()

# Convert diagnosis to model class names
diagnosis_to_class = {
    "melanoma": "Melanoma",
    "basal cell carcinoma": "Basal Cell Carcinoma",
    "squamous cell carcinoma": "Squamous Cell Carcinoma"
}

isic_3class["class_name"] = (
    isic_3class["diagnosis"]
    .map(diagnosis_to_class)
)

# Numeric labels
class_to_label = {
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2
}

isic_3class["label"] = (
    isic_3class["class_name"]
    .map(class_to_label)
)

print("\n3-class distribution:")
print(isic_3class["class_name"].value_counts())

print("\nTotal 3-class images:", len(isic_3class))

print("\n=======================================")
print("ISIC 3-CLASS DATA READY")
print("=======================================")

In [ ]:
# =======================================
# RELOAD ISIC METADATA
# =======================================

import pandas as pd
import os

isic_path = "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/metadata.csv"

isic = pd.read_csv(
    isic_path,
    low_memory=False
)

# Recreate image paths
image_root = "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/images"

isic["image_path"] = isic["isic_id"].apply(
    lambda x: os.path.join(image_root, x + ".jpg")
)

print("=======================================")
print("ISIC METADATA RELOADED")
print("=======================================")

print("Total ISIC records:", len(isic))
print("Columns:", isic.columns.tolist())

print("\n=======================================")
print("ISIC METADATA READY")
print("=======================================")

In [ ]:
# =======================================
# RECREATE ISIC 3-CLASS DATA
# =======================================

print("=======================================")
print("RECREATING ISIC 3-CLASS DATA")
print("=======================================")

target_diagnoses = [
    "melanoma",
    "basal cell carcinoma",
    "squamous cell carcinoma"
]

isic_3class = isic[
    isic["diagnosis"].isin(target_diagnoses)
].copy()

diagnosis_to_class = {
    "melanoma": "Melanoma",
    "basal cell carcinoma": "Basal Cell Carcinoma",
    "squamous cell carcinoma": "Squamous Cell Carcinoma"
}

isic_3class["class_name"] = (
    isic_3class["diagnosis"]
    .map(diagnosis_to_class)
)

class_to_label = {
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2
}

isic_3class["label"] = (
    isic_3class["class_name"]
    .map(class_to_label)
)

print("\n=======================================")
print("3-CLASS DISTRIBUTION")
print("=======================================")

print(
    isic_3class["class_name"].value_counts()
)

print(
    "\nTotal 3-class images:",
    len(isic_3class)
)

print("\n=======================================")
print("ISIC 3-CLASS DATA READY")
print("=======================================")

In [ ]:
# =======================================
# CREATE FINAL 4-CLASS DATASET
# =======================================

print("=======================================")
print("CREATING FINAL 4-CLASS DATASET")
print("=======================================")

# Keep only required columns
isic_final = isic_3class[
    ["image_path", "class_name", "label"]
].copy()

mcc_final = mcc_df[
    ["image_path", "class_name", "label"]
].copy()

# Combine ISIC + MCC
final_dataset = pd.concat(
    [isic_final, mcc_final],
    ignore_index=True
)

# Shuffle dataset
final_dataset = final_dataset.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\n=======================================")
print("FINAL 4-CLASS DISTRIBUTION")
print("=======================================")

print(
    final_dataset["class_name"].value_counts()
)

print(
    "\nTotal images:",
    len(final_dataset)
)

print("\nLabel mapping:")
print("0 = Melanoma")
print("1 = Basal Cell Carcinoma")
print("2 = Squamous Cell Carcinoma")
print("3 = Merkel Cell Carcinoma")

print("\n=======================================")
print("FINAL 4-CLASS DATASET READY")
print("=======================================")

In [18]:
# =======================================
# STRATIFIED TRAIN / VALIDATION SPLIT
# =======================================

from sklearn.model_selection import train_test_split

print("=======================================")
print("CREATING TRAIN / VALIDATION SPLIT")
print("=======================================")

train_df, val_df = train_test_split(
    combined_df,
    test_size=0.20,
    stratify=combined_df["label"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("\nTrain size:", len(train_df))
print("Validation size:", len(val_df))

print("\n=======================================")
print("TRAIN CLASS DISTRIBUTION")
print("=======================================")

print(train_df["class_name"].value_counts())

print("\n=======================================")
print("VALIDATION CLASS DISTRIBUTION")
print("=======================================")

print(val_df["class_name"].value_counts())

print("\n=======================================")
print("STRATIFIED SPLIT COMPLETE")
print("=======================================")

CREATING TRAIN / VALIDATION SPLIT

Train size: 12089
Validation size: 3023

TRAIN CLASS DISTRIBUTION
class_name
Melanoma                   5879
Basal Cell Carcinoma       3937
Squamous Cell Carcinoma    2260
Merkel Cell Carcinoma        13
Name: count, dtype: int64

VALIDATION CLASS DISTRIBUTION
class_name
Melanoma                   1470
Basal Cell Carcinoma        984
Squamous Cell Carcinoma     565
Merkel Cell Carcinoma         4
Name: count, dtype: int64

STRATIFIED SPLIT COMPLETE


In [19]:
# =======================================
# CHECK TRANSFORMS
# =======================================

print("train_transform exists:", "train_transform" in globals())
print("test_transform exists:", "test_transform" in globals())

train_transform exists: False
test_transform exists: False


In [20]:
# =======================================
# IMAGE TRANSFORMS
# =======================================

from torchvision import transforms

# Training images
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Validation images
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("=======================================")
print("IMAGE TRANSFORMS READY")
print("=======================================")

print("Train transform:", train_transform)
print("Validation transform:", test_transform)

IMAGE TRANSFORMS READY
Train transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Validation transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [21]:
# =======================================
# SKIN CANCER DATASET CLASS
# =======================================

from PIL import Image
from torch.utils.data import Dataset

class SkinCancerDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image_path = row["image_path"]
        label = int(row["label"])

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(
                f"Could not load image: {image_path}"
            ) from e

        if self.transform:
            image = self.transform(image)

        return image, label


print("=======================================")
print("DATASET CLASS READY")
print("=======================================")

print("Dataset class:", SkinCancerDataset)

DATASET CLASS READY
Dataset class: <class '__main__.SkinCancerDataset'>


In [ ]:
# =======================================
# CREATING FINAL 4-CLASS DATALOADERS
# =======================================

from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch

print("=======================================")
print("CREATING FINAL 4-CLASS DATALOADERS")
print("=======================================")


class SkinCancerDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        label = int(row["label"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label


# Create datasets
train_dataset = SkinCancerDataset(
    train_df,
    transform=train_transform
)

val_dataset = SkinCancerDataset(
    val_df,
    transform=test_transform
)


# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=False
)


print("\nTrain dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

print("\nTrain batches:", len(train_loader))
print("Validation batches:", len(val_loader))

print("\n=======================================")
print("FINAL 4-CLASS DATALOADERS READY")
print("=======================================")

In [ ]:
# =======================================
# RECREATE IMAGE TRANSFORMS
# =======================================

from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("=======================================")
print("IMAGE TRANSFORMS READY")
print("=======================================")
print("Train transform: READY")
print("Test transform : READY")

In [ ]:
=======================================
CREATING FINAL 4-CLASS DATALOADERS
=======================================

Train dataset: 10924
Validation dataset: 2732

Train batches: 342
Validation batches: 86

=======================================
FINAL 4-CLASS DATALOADERS READY
=======================================

In [ ]:
# =======================================
# CREATE FINAL 4-CLASS MODEL
# =======================================

import torch
import torch.nn as nn
from torchvision import models

print("=======================================")
print("CREATING 4-CLASS MODEL")
print("=======================================")

# Use CPU because current PyTorch CUDA is incompatible
device = torch.device("cpu")

print("Device:", device)

# Load ResNet18
model = models.resnet18(weights="DEFAULT")

# Replace final layer for 4 classes
model.fc = nn.Linear(
    model.fc.in_features,
    4
)

# Move model to CPU
model = model.to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("\nModel:")
print(model.fc)

print("\nClass mapping:")
print("0 = Melanoma")
print("1 = Basal Cell Carcinoma")
print("2 = Squamous Cell Carcinoma")
print("3 = Merkel Cell Carcinoma")

print("\n=======================================")
print("4-CLASS MODEL SETUP COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# TEST MODEL + DATA BATCH
# =======================================

print("=======================================")
print("TESTING MODEL + DATA BATCH")
print("=======================================")

model.eval()

images, labels = next(iter(train_loader))

with torch.no_grad():

    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)
    loss = criterion(outputs, labels)

print("\nInput shape :", images.shape)
print("Output shape:", outputs.shape)
print("Labels shape:", labels.shape)

print("\nTest loss:", loss.item())

print("\n=======================================")
print("MODEL + BATCH TEST SUCCESSFUL")
print("=======================================")

In [ ]:
# =======================================
# RECREATE FINAL 4-CLASS DATALOADERS
# =======================================

from torch.utils.data import Dataset, DataLoader
from PIL import Image

print("=======================================")
print("CREATING FINAL 4-CLASS DATALOADERS")
print("=======================================")


class SkinCancerDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        label = int(row["label"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label


train_dataset = SkinCancerDataset(
    train_df,
    transform=train_transform
)

val_dataset = SkinCancerDataset(
    val_df,
    transform=test_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=False
)


print("\nTrain dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

print("\nTrain batches:", len(train_loader))
print("Validation batches:", len(val_loader))

print("\n=======================================")
print("FINAL 4-CLASS DATALOADERS READY")
print("=======================================")

In [ ]:
# =======================================
# SAVE CURRENT MODEL CHECKPOINT
# =======================================

import torch

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "device": str(device),
    "class_mapping": {
        0: "Melanoma",
        1: "Basal Cell Carcinoma",
        2: "Squamous Cell Carcinoma",
        3: "Merkel Cell Carcinoma"
    }
}

torch.save(checkpoint, "/kaggle/working/4class_resnet18_checkpoint.pth")

print("=======================================")
print("CHECKPOINT SAVED SUCCESSFULLY")
print("=======================================")
print("File:")
print("/kaggle/working/4class_resnet18_checkpoint.pth")

In [ ]:
# =======================================
# TEST MODEL FORWARD PASS
# =======================================

print("=======================================")
print("TESTING MODEL FORWARD PASS")
print("=======================================")

model.eval()

with torch.no_grad():
    test_images = images.to(device)
    outputs = model(test_images)

print("Input shape :", test_images.shape)
print("Output shape:", outputs.shape)

print("\nModel output:")
print(outputs)

print("\n=======================================")
print("MODEL FORWARD PASS COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# TRAIN / TEST TRANSFORMS
# =======================================

from torchvision import transforms

print("=======================================")
print("CREATING IMAGE TRANSFORMS")
print("=======================================")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Train transform READY")
print("Test transform READY")

print("=======================================")
print("IMAGE TRANSFORMS READY")
print("=======================================")

In [ ]:
# =======================================
# CREATE CLASS NAMES FROM ISIC METADATA
# =======================================

print("=======================================")
print("ISIC COLUMNS")
print("=======================================")

print(isic.columns.tolist())

# Check diagnosis column
print("\n=======================================")
print("DIAGNOSIS VALUES")
print("=======================================")

print(isic["diagnosis"].value_counts(dropna=False).head(20))

In [ ]:
# =======================================
# CREATE 4-CLASS ISIC DATA
# =======================================

import pandas as pd

# Create class_name from diagnosis
def assign_class(diagnosis):
    if pd.isna(diagnosis):
        return None
    
    diagnosis = str(diagnosis).strip().lower()
    
    if diagnosis == "melanoma":
        return "Melanoma"
    
    elif diagnosis == "basal cell carcinoma":
        return "Basal Cell Carcinoma"
    
    elif diagnosis == "squamous cell carcinoma":
        return "Squamous Cell Carcinoma"
    
    else:
        return None


isic["class_name"] = isic["diagnosis"].apply(assign_class)

# Keep only our 3 cancer classes
isic_clean = isic[
    isic["class_name"].notna()
].copy()

# Assign labels
isic_clean["label"] = isic_clean["class_name"].map({
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2
})

isic_clean["dataset"] = "ISIC"

# Create image_name
isic_clean["image_name"] = isic_clean["isic_id"].astype(str) + ".jpg"

# Keep required columns
isic_clean = isic_clean[
    [
        "image_name",
        "image_path",
        "class_name",
        "label",
        "dataset"
    ]
].copy()

# Remove missing paths
isic_clean = isic_clean[
    isic_clean["image_path"].notna()
].copy()

# Remove duplicate image paths
isic_clean = isic_clean.drop_duplicates(
    subset=["image_path"]
).reset_index(drop=True)


print("=======================================")
print("ISIC 3-CLASS DATA READY")
print("=======================================")

print("Total images:", len(isic_clean))

print("\nClass distribution:")
print(isic_clean["class_name"].value_counts())

print("\nLabel distribution:")
print(isic_clean["label"].value_counts().sort_index())

print("\nExample:")
print(isic_clean.head())

In [ ]:
# =======================================
# PREPARE MCC DATA
# =======================================

import os
import glob
import pandas as pd

mcc_dir = "/kaggle/input/datasets/quantumcoders05/mccdataset/mcc image"

# Find MCC images
mcc_files = glob.glob(
    os.path.join(mcc_dir, "*")
)

mcc_files = [
    p for p in mcc_files
    if p.lower().endswith(
        (".jpg", ".jpeg", ".png", ".webp")
    )
]

# Create MCC dataframe
new_mcc_df = pd.DataFrame({
    "image_name": [
        os.path.basename(p) for p in mcc_files
    ],
    "image_path": mcc_files,
    "class_name": "Merkel Cell Carcinoma",
    "label": 3,
    "dataset": "NEW_MCC"
})

print("=======================================")
print("MCC DATA READY")
print("=======================================")

print("MCC images:", len(new_mcc_df))

print("\nClass distribution:")
print(new_mcc_df["class_name"].value_counts())

print("\nExample images:")
print(new_mcc_df.head())

In [ ]:
# =======================================
# CREATE FINAL 4-CLASS DATASET
# =======================================

# ISIC data
isic_final = isic_clean[
    ["image_name", "image_path", "class_name", "label", "dataset"]
].copy()

# MCC data
mcc_final = new_mcc_df[
    ["image_name", "image_path", "class_name", "label", "dataset"]
].copy()

# Combine
combined_df = pd.concat(
    [isic_final, mcc_final],
    ignore_index=True
)

# Remove missing paths
combined_df = combined_df[
    combined_df["image_path"].notna()
].copy()

# Remove duplicate image paths
combined_df = combined_df.drop_duplicates(
    subset=["image_path"]
).reset_index(drop=True)

print("=======================================")
print("FINAL 4-CLASS DATASET")
print("=======================================")

print("Total images:", len(combined_df))

print("\nCLASS DISTRIBUTION")
print(combined_df["class_name"].value_counts())

print("\nLABEL DISTRIBUTION")
print(combined_df["label"].value_counts().sort_index())

print("\nDATASET DISTRIBUTION")
print(combined_df["dataset"].value_counts())

print("\nLABEL MAPPING")
print("0 = Melanoma")
print("1 = Basal Cell Carcinoma")
print("2 = Squamous Cell Carcinoma")
print("3 = Merkel Cell Carcinoma")

In [ ]:
# =======================================
# REBUILD FINAL 4-CLASS DATASET
# =======================================

import pandas as pd
import os

print("=======================================")
print("REBUILDING FINAL 4-CLASS DATASET")
print("=======================================")

# ---------------------------------------
# 1. Prepare ISIC data
# ---------------------------------------

isic_clean = isic[
    isic["diagnosis"].isin([
        "melanoma",
        "basal cell carcinoma",
        "squamous cell carcinoma"
    ])
].copy()

# Create class names
def make_class_name(x):

    if x == "melanoma":
        return "Melanoma"

    elif x == "basal cell carcinoma":
        return "Basal Cell Carcinoma"

    elif x == "squamous cell carcinoma":
        return "Squamous Cell Carcinoma"

    return None


isic_clean["class_name"] = isic_clean["diagnosis"].apply(
    make_class_name
)

# Create labels
label_map = {
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2,
    "Merkel Cell Carcinoma": 3
}

isic_clean["label"] = isic_clean["class_name"].map(
    label_map
)

# Image name
isic_clean["image_name"] = isic_clean["image_path"].apply(
    lambda x: os.path.basename(str(x))
)

isic_clean["dataset"] = "ISIC"

isic_clean = isic_clean[
    [
        "image_name",
        "image_path",
        "class_name",
        "label",
        "dataset"
    ]
].copy()


# ---------------------------------------
# 2. Prepare SCC data
# ---------------------------------------

scc_clean = new_scc_df[
    [
        "image_name",
        "image_path",
        "class_name",
        "label",
        "dataset"
    ]
].copy()


# ---------------------------------------
# 3. Prepare MCC data
# ---------------------------------------

mcc_clean = new_mcc_df[
    [
        "image_name",
        "image_path",
        "class_name",
        "label",
        "dataset"
    ]
].copy()


# ---------------------------------------
# 4. Combine
# ---------------------------------------

combined_df = pd.concat(
    [
        isic_clean,
        scc_clean,
        mcc_clean
    ],
    ignore_index=True
)


# ---------------------------------------
# 5. Remove missing paths
# ---------------------------------------

combined_df = combined_df[
    combined_df["image_path"].notna()
].copy()


# ---------------------------------------
# 6. Remove duplicate image paths
# ---------------------------------------

before = len(combined_df)

combined_df = combined_df.drop_duplicates(
    subset=["image_path"]
).reset_index(drop=True)

duplicates_removed = before - len(combined_df)


# ---------------------------------------
# 7. Final output
# ---------------------------------------

print("\n=======================================")
print("FINAL 4-CLASS DATASET READY")
print("=======================================")

print("Total samples:", len(combined_df))

print("Duplicates removed:", duplicates_removed)

print("\nCLASS DISTRIBUTION")
print("---------------------------------------")

print(
    combined_df["class_name"].value_counts()
)

print("\nLABEL DISTRIBUTION")
print("---------------------------------------")

print(
    combined_df["label"]
    .value_counts()
    .sort_index()
)

print("\nDATASET DISTRIBUTION")
print("---------------------------------------")

print(
    combined_df["dataset"].value_counts()
)

print("\n=======================================")
print("CLASS COUNTS")
print("=======================================")

for class_name in [
    "Melanoma",
    "Basal Cell Carcinoma",
    "Squamous Cell Carcinoma",
    "Merkel Cell Carcinoma"
]:

    count = (
        combined_df["class_name"] == class_name
    ).sum()

    print(f"{class_name}: {count}")


print("\n=======================================")
print("LABEL MAPPING")
print("=======================================")

print("0 = Melanoma")
print("1 = Basal Cell Carcinoma")
print("2 = Squamous Cell Carcinoma")
print("3 = Merkel Cell Carcinoma")

In [ ]:
# ============================================================
# REBUILD EVERYTHING FROM SCRATCH
# ISIC + SCC + MCC -> FINAL 4-CLASS DATASET
# ============================================================

import os
import glob
import pandas as pd

print("=======================================")
print("REBUILDING FINAL 4-CLASS DATASET")
print("=======================================")


# ============================================================
# 1. LOAD ISIC METADATA
# ============================================================

isic_metadata_path = (
    "/kaggle/input/datasets/tomooinubushi/"
    "all-isic-data-20240629/metadata.csv"
)

isic = pd.read_csv(
    isic_metadata_path,
    low_memory=False
)

print("\nISIC metadata loaded:", len(isic))


# ============================================================
# 2. FIND ISIC IMAGE FILES
# ============================================================

isic_image_dir = (
    "/kaggle/input/datasets/tomooinubushi/"
    "all-isic-data-20240629/images"
)

isic_image_files = glob.glob(
    os.path.join(isic_image_dir, "**", "*"),
    recursive=True
)

isic_image_files = [
    p for p in isic_image_files
    if p.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("ISIC image files:", len(isic_image_files))


# ============================================================
# 3. CREATE ISIC IMAGE PATH MAP
# ============================================================

isic_image_map = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in isic_image_files
}


# ============================================================
# 4. MATCH ISIC IMAGE PATHS
# ============================================================

def get_isic_image_path(row):

    image_id = str(row["isic_id"])

    return isic_image_map.get(image_id, None)


isic["image_path"] = isic.apply(
    get_isic_image_path,
    axis=1
)

print("\nISIC IMAGE PATH MATCHING")
print("---------------------------------------")
print("Total ISIC records:", len(isic))
print(
    "Valid image paths:",
    isic["image_path"].notna().sum()
)
print(
    "Missing image paths:",
    isic["image_path"].isna().sum()
)


# ============================================================
# 5. CREATE ISIC 3 MAIN CLASSES
# ============================================================

isic_clean = isic[
    isic["diagnosis"].isin([
        "melanoma",
        "basal cell carcinoma"
    ])
].copy()

# Create class names
isic_clean["class_name"] = isic_clean["diagnosis"].map({

    "melanoma":
        "Melanoma",

    "basal cell carcinoma":
        "Basal Cell Carcinoma"

})

# Create labels
isic_clean["label"] = isic_clean["class_name"].map({

    "Melanoma": 0,

    "Basal Cell Carcinoma": 1

})

isic_clean["dataset"] = "ISIC"

isic_clean["image_name"] = isic_clean[
    "image_path"
].apply(
    lambda x: os.path.basename(str(x))
    if pd.notna(x) else None
)

isic_clean = isic_clean[
    [
        "image_name",
        "image_path",
        "class_name",
        "label",
        "dataset"
    ]
].copy()

print("\nISIC MAIN CLASSES")
print("---------------------------------------")
print(
    isic_clean["class_name"].value_counts()
)


# ============================================================
# 6. LOAD NEW SCC DATA
# ============================================================

scc_base = (
    "/kaggle/input/datasets/"
    "riyaelizashaju/"
    "isic-skin-disease-image-dataset-labelled/"
    "ISIC_Labelled/"
    "Squamous cell carcinoma"
)

scc_files = glob.glob(
    os.path.join(scc_base, "*")
)

scc_files = [
    p for p in scc_files
    if p.lower().endswith((".jpg", ".jpeg", ".png"))
]


# ============================================================
# 7. ADD SCC FROM SKIN_DS
# ============================================================

skin_ds_base = (
    "/kaggle/input/datasets/"
    "ahmedxc4/skin-ds"
)

skin_ds_scc_files = []

for split in ["train", "val", "test"]:

    folder = os.path.join(
        skin_ds_base,
        split,
        "Squamous cell carcinoma"
    )

    if os.path.exists(folder):

        files = glob.glob(
            os.path.join(folder, "*")
        )

        files = [
            p for p in files
            if p.lower().endswith(
                (".jpg", ".jpeg", ".png")
            )
        ]

        skin_ds_scc_files.extend(files)


# ============================================================
# 8. ADD SCC FROM SKIN CANCER 9
# ============================================================

skin9_base = (
    "/kaggle/input/datasets/"
    "nodoubttome/"
    "skin-cancer9-classesisic/"
)

skin9_scc_files = []

for root, dirs, files in os.walk(skin9_base):

    if os.path.basename(root).lower() == \
       "squamous cell carcinoma":

        for file in files:

            if file.lower().endswith(
                (".jpg", ".jpeg", ".png")
            ):

                skin9_scc_files.append(
                    os.path.join(root, file)
                )


# ============================================================
# 9. COMBINE SCC FILES
# ============================================================

all_scc_files = (
    scc_files
    + skin_ds_scc_files
    + skin9_scc_files
)

print("\nSCC FILES FOUND")
print("---------------------------------------")
print("SCC images:", len(all_scc_files))


new_scc_df = pd.DataFrame({

    "image_name": [
        os.path.basename(p)
        for p in all_scc_files
    ],

    "image_path": all_scc_files,

    "class_name": [
        "Squamous Cell Carcinoma"
    ] * len(all_scc_files),

    "label": [2] * len(all_scc_files),

    "dataset": ["NEW_SCC"] * len(all_scc_files)

})


# ============================================================
# 10. LOAD MCC
# ============================================================

mcc_base = (
    "/kaggle/input/datasets/"
    "quantumcoders05/"
    "mccdataset/"
    "mcc image"
)

mcc_files = glob.glob(
    os.path.join(mcc_base, "*")
)

mcc_files = [
    p for p in mcc_files
    if p.lower().endswith(
        (".jpg", ".jpeg", ".png", ".webp")
    )
]

print("\nMCC FILES FOUND")
print("---------------------------------------")
print("MCC images:", len(mcc_files))


new_mcc_df = pd.DataFrame({

    "image_name": [
        os.path.basename(p)
        for p in mcc_files
    ],

    "image_path": mcc_files,

    "class_name": [
        "Merkel Cell Carcinoma"
    ] * len(mcc_files),

    "label": [3] * len(mcc_files),

    "dataset": ["NEW_MCC"] * len(mcc_files)

})


# ============================================================
# 11. COMBINE ALL FOUR CLASSES
# ============================================================

combined_df = pd.concat(
    [
        isic_clean,
        new_scc_df,
        new_mcc_df
    ],
    ignore_index=True
)


# ============================================================
# 12. REMOVE MISSING PATHS
# ============================================================

combined_df = combined_df[
    combined_df["image_path"].notna()
].copy()


# ============================================================
# 13. REMOVE EXACT DUPLICATE PATHS
# ============================================================

before = len(combined_df)

combined_df = combined_df.drop_duplicates(
    subset=["image_path"]
).reset_index(drop=True)

duplicates_removed = (
    before - len(combined_df)
)


# ============================================================
# 14. FINAL RESULTS
# ============================================================

print("\n=======================================")
print("FINAL 4-CLASS DATASET READY")
print("=======================================")

print(
    "Total samples:",
    len(combined_df)
)

print(
    "Duplicate paths removed:",
    duplicates_removed
)

print("\nCLASS DISTRIBUTION")
print("---------------------------------------")

print(
    combined_df["class_name"].value_counts()
)

print("\nLABEL DISTRIBUTION")
print("---------------------------------------")

print(
    combined_df["label"]
    .value_counts()
    .sort_index()
)

print("\nDATASET DISTRIBUTION")
print("---------------------------------------")

print(
    combined_df["dataset"].value_counts()
)

print("\nIMAGE PATH CHECK")
print("---------------------------------------")

print(
    "Missing paths:",
    combined_df["image_path"].isna().sum()
)

print("\n=======================================")
print("CLASS COUNTS")
print("=======================================")

for class_name in [
    "Melanoma",
    "Basal Cell Carcinoma",
    "Squamous Cell Carcinoma",
    "Merkel Cell Carcinoma"
]:

    count = (
        combined_df["class_name"]
        == class_name
    ).sum()

    print(
        f"{class_name}: {count}"
    )

print("\n=======================================")
print("LABEL MAPPING")
print("=======================================")

print("0 = Melanoma")
print("1 = Basal Cell Carcinoma")
print("2 = Squamous Cell Carcinoma")
print("3 = Merkel Cell Carcinoma")

In [ ]:
# ============================================================
# CHECK AND REMOVE CROSS-DATASET IMAGE DUPLICATES
# ============================================================

print("=======================================")
print("CROSS-DATASET DUPLICATE CHECK")
print("=======================================")

# ------------------------------------------------------------
# Check duplicate image names
# ------------------------------------------------------------

duplicate_names = combined_df[
    combined_df["image_name"].duplicated(
        keep=False
    )
].sort_values("image_name")

print("\nDuplicate image names:", len(duplicate_names))

if len(duplicate_names) > 0:

    print("\nExamples of duplicate image names:")
    print(
        duplicate_names[
            [
                "image_name",
                "class_name",
                "dataset"
            ]
        ].head(20)
    )

else:

    print("No duplicate image names found.")


# ------------------------------------------------------------
# Check duplicates specifically between ISIC and NEW_SCC
# ------------------------------------------------------------

isic_names = set(
    combined_df.loc[
        combined_df["dataset"] == "ISIC",
        "image_name"
    ]
)

new_scc_names = set(
    combined_df.loc[
        combined_df["dataset"] == "NEW_SCC",
        "image_name"
    ]
)

overlap_scc = isic_names.intersection(
    new_scc_names
)

print("\n=======================================")
print("ISIC vs NEW_SCC OVERLAP")
print("=======================================")

print(
    "Overlapping image names:",
    len(overlap_scc)
)

if len(overlap_scc) > 0:

    print("\nExamples:")
    print(
        list(overlap_scc)[:20]
    )


# ------------------------------------------------------------
# Remove NEW_SCC images already present in ISIC
# ------------------------------------------------------------

before_count = len(combined_df)

combined_df = combined_df[
    ~(
        (combined_df["dataset"] == "NEW_SCC")
        &
        (combined_df["image_name"].isin(isic_names))
    )
].copy()

combined_df = combined_df.reset_index(
    drop=True
)

removed_count = (
    before_count - len(combined_df)
)


# ------------------------------------------------------------
# Final duplicate check
# ------------------------------------------------------------

print("\n=======================================")
print("CLEAN DATASET")
print("=======================================")

print(
    "Images removed due to cross-dataset duplication:",
    removed_count
)

print(
    "Final total images:",
    len(combined_df)
)

print("\nCLASS DISTRIBUTION")
print("---------------------------------------")

print(
    combined_df["class_name"].value_counts()
)

print("\nDATASET DISTRIBUTION")
print("---------------------------------------")

print(
    combined_df["dataset"].value_counts()
)

print("\nDUPLICATE IMAGE NAME CHECK")
print("---------------------------------------")

print(
    "Duplicate image names:",
    combined_df["image_name"].duplicated().sum()
)

print("\nDUPLICATE IMAGE PATH CHECK")
print("---------------------------------------")

print(
    "Duplicate image paths:",
    combined_df["image_path"].duplicated().sum()
)

In [ ]:
# ============================================================
# REMOVE DUPLICATE IMAGE NAMES
# ============================================================

print("=======================================")
print("REMOVING DUPLICATE IMAGE NAMES")
print("=======================================")

# Check duplicate rows before removal
before = len(combined_df)

duplicate_count = combined_df["image_name"].duplicated(
    keep="first"
).sum()

print("Duplicate image names found:", duplicate_count)

# ------------------------------------------------------------
# Remove duplicate image names
# Keep only the first occurrence
# ------------------------------------------------------------

combined_df = combined_df.drop_duplicates(
    subset=["image_name"],
    keep="first"
).reset_index(drop=True)

removed = before - len(combined_df)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

print("\n=======================================")
print("DUPLICATES REMOVED")
print("=======================================")

print("Rows before:", before)
print("Rows removed:", removed)
print("Rows after:", len(combined_df))

print("\n=======================================")
print("FINAL CLASS DISTRIBUTION")
print("=======================================")

print(
    combined_df["class_name"].value_counts()
)

print("\n=======================================")
print("FINAL DATASET DISTRIBUTION")
print("=======================================")

print(
    combined_df["dataset"].value_counts()
)

print("\n=======================================")
print("DUPLICATE CHECK")
print("=======================================")

print(
    "Duplicate image names:",
    combined_df["image_name"].duplicated().sum()
)

print(
    "Duplicate image paths:",
    combined_df["image_path"].duplicated().sum()
)

print("\n=======================================")
print("FINAL CLASS COUNTS")
print("=======================================")

for class_name in [
    "Melanoma",
    "Basal Cell Carcinoma",
    "Squamous Cell Carcinoma",
    "Merkel Cell Carcinoma"
]:

    count = (
        combined_df["class_name"] == class_name
    ).sum()

    print(f"{class_name}: {count}")

In [ ]:
# ============================================================
# FINAL STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

print("=======================================")
print("CREATING FINAL DATASET SPLIT")
print("=======================================")

# ------------------------------------------------------------
# First split: 80% train, 20% temporary
# ------------------------------------------------------------

train_df, temp_df = train_test_split(
    combined_df,
    test_size=0.20,
    stratify=combined_df["label"],
    random_state=42
)

# ------------------------------------------------------------
# Second split:
# 10% validation + 10% test
# ------------------------------------------------------------

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

# Reset indexes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n=======================================")
print("FINAL DATASET SPLIT")
print("=======================================")

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

# ------------------------------------------------------------
# Training distribution
# ------------------------------------------------------------

print("\n=======================================")
print("TRAIN DISTRIBUTION")
print("=======================================")

print(
    train_df["class_name"].value_counts()
)

# ------------------------------------------------------------
# Validation distribution
# ------------------------------------------------------------

print("\n=======================================")
print("VALIDATION DISTRIBUTION")
print("=======================================")

print(
    val_df["class_name"].value_counts()
)

# ------------------------------------------------------------
# Test distribution
# ------------------------------------------------------------

print("\n=======================================")
print("TEST DISTRIBUTION")
print("=======================================")

print(
    test_df["class_name"].value_counts()
)

# ------------------------------------------------------------
# Label distributions
# ------------------------------------------------------------

print("\n=======================================")
print("LABEL DISTRIBUTIONS")
print("=======================================")

print("\nTrain:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation:")
print(val_df["label"].value_counts().sort_index())

print("\nTest:")
print(test_df["label"].value_counts().sort_index())

# ------------------------------------------------------------
# Check for overlap between splits
# ------------------------------------------------------------

train_paths = set(train_df["image_path"])
val_paths = set(val_df["image_path"])
test_paths = set(test_df["image_path"])

print("\n=======================================")
print("DATA LEAKAGE CHECK")
print("=======================================")

print(
    "Train ∩ Validation:",
    len(train_paths.intersection(val_paths))
)

print(
    "Train ∩ Test:",
    len(train_paths.intersection(test_paths))
)

print(
    "Validation ∩ Test:",
    len(val_paths.intersection(test_paths))
)

In [ ]:
import torch

print("=======================================")
print("FINAL PYTORCH + GPU CHECK")
print("=======================================")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))

    try:
        x = torch.randn(2, 3, 224, 224).cuda()
        y = x * 2
        torch.cuda.synchronize()

        print("GPU TEST: PASSED")

    except Exception as e:
        print("GPU TEST: FAILED")
        print("Error:", e)

else:
    print("GPU TEST: CUDA NOT AVAILABLE")

In [ ]:
# ============================================================
# CREATE 4-CLASS IMAGE DATASETS + DATALOADERS
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

print("=======================================")
print("CREATING 4-CLASS DATALOADERS")
print("=======================================")

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# IMAGE TRANSFORMS
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomVerticalFlip(
        p=0.2
    ),

    transforms.RandomRotation(
        15
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# CUSTOM DATASET
# ============================================================

class SkinCancerDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = row["image_path"]
        label = int(row["label"])

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            print(
                "Error loading:",
                image_path
            )

            raise e

        if self.transform is not None:

            image = self.transform(
                image
            )

        return image, label


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = SkinCancerDataset(
    train_df,
    transform=train_transform
)

val_dataset = SkinCancerDataset(
    val_df,
    transform=eval_transform
)

test_dataset = SkinCancerDataset(
    test_df,
    transform=eval_transform
)


# ============================================================
# CREATE DATALOADERS
# ============================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# CHECK
# ============================================================

print("\n=======================================")
print("DATALOADERS CREATED")
print("=======================================")

print(
    "Training samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)

print(
    "Testing samples:",
    len(test_dataset)
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Testing batches:",
    len(test_loader)
)

print(
    "Batch size:",
    BATCH_SIZE
)


# ============================================================
# TEST ONE BATCH
# ============================================================

images, labels = next(
    iter(train_loader)
)

print("\n=======================================")
print("BATCH TEST")
print("=======================================")

print(
    "Image batch shape:",
    images.shape
)

print(
    "Label batch shape:",
    labels.shape
)

print(
    "Labels in batch:",
    labels.tolist()[:20]
)

print(
    "Image tensor device:",
    images.device
)


# ============================================================
# GPU TEST WITH BATCH
# ============================================================

try:

    images_gpu = images.to(
        device,
        non_blocking=True
    )

    labels_gpu = labels.to(
        device,
        non_blocking=True
    )

    torch.cuda.synchronize()

    print("\nGPU BATCH TEST: PASSED")

    print(
        "GPU image tensor:",
        images_gpu.shape
    )

    print(
        "GPU label tensor:",
        labels_gpu.shape
    )

except Exception as e:

    print("\nGPU BATCH TEST: FAILED")
    print(e)

In [ ]:
# ========================================
# TRAIN 4-CLASS RESNET18
# ========================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import os

print("=======================================")
print("STARTING 4-CLASS RESNET18 TRAINING")
print("=======================================")

# ----------------------------------------
# Device
# ----------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ----------------------------------------
# Class names
# ----------------------------------------

class_names = [
    "Melanoma",
    "Basal Cell Carcinoma",
    "Squamous Cell Carcinoma",
    "Merkel Cell Carcinoma"
]

num_classes = 4


# ----------------------------------------
# Create ResNet18
# ----------------------------------------

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace final layer
model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

model = model.to(device)


print("\n=======================================")
print("MODEL")
print("=======================================")

print(model.fc)


# ----------------------------------------
# Calculate class weights
# ----------------------------------------

train_labels = np.array(train_df["label"])

class_counts = np.bincount(
    train_labels,
    minlength=num_classes
)

print("\n=======================================")
print("CLASS COUNTS")
print("=======================================")

for i, count in enumerate(class_counts):
    print(f"{i} = {class_names[i]}: {count}")


# ----------------------------------------
# Balanced class weights
# ----------------------------------------

class_weights = len(train_labels) / (
    num_classes * class_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("\nClass weights:")
print(class_weights)


# ----------------------------------------
# Loss function
# ----------------------------------------

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ----------------------------------------
# Optimizer
# ----------------------------------------

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0001
)


# ----------------------------------------
# Scheduler
# ----------------------------------------

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


# ----------------------------------------
# Training settings
# ----------------------------------------

num_epochs = 15

best_val_accuracy = 0.0

best_model_path = "/kaggle/working/best_resnet18_4class.pth"


# ========================================
# TRAINING LOOP
# ========================================

for epoch in range(num_epochs):

    print("\n")
    print("=======================================")
    print(f"EPOCH {epoch + 1}/{num_epochs}")
    print("=======================================")

    # ------------------------------------
    # TRAIN
    # ------------------------------------

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients
        optimizer.zero_grad()

        # Forward
        outputs = model(images)

        # Loss
        loss = criterion(outputs, labels)

        # Backward
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

    train_loss = running_loss / total
    train_accuracy = 100.0 * correct / total


    # ------------------------------------
    # VALIDATION
    # ------------------------------------

    model.eval()

    val_loss_total = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss_total += (
                loss.item() * images.size(0)
            )

            _, predicted = torch.max(
                outputs,
                1
            )

            val_total += labels.size(0)

            val_correct += (
                predicted == labels
            ).sum().item()

    val_loss = val_loss_total / val_total

    val_accuracy = (
        100.0 * val_correct / val_total
    )


    # ------------------------------------
    # Scheduler
    # ------------------------------------

    scheduler.step(val_accuracy)


    # ------------------------------------
    # Print results
    # ------------------------------------

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Train Accuracy: {train_accuracy:.2f}%"
    )

    print(
        f"Val Loss: {val_loss:.4f}"
    )

    print(
        f"Val Accuracy: {val_accuracy:.2f}%"
    )

    print(
        f"Learning Rate: {current_lr:.7f}"
    )


    # ------------------------------------
    # Save best model
    # ------------------------------------

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_accuracy": val_accuracy,
                "class_names": class_names
            },
            best_model_path
        )

        print("\n*** BEST MODEL SAVED ***")
        print(
            f"Best Validation Accuracy: "
            f"{best_val_accuracy:.2f}%"
        )


# ========================================
# TRAINING COMPLETE
# ========================================

print("\n")
print("=======================================")
print("TRAINING COMPLETE")
print("=======================================")

print(
    f"Best Validation Accuracy: "
    f"{best_val_accuracy:.2f}%"
)

print(
    "Best model saved at:"
)

print(best_model_path)

In [ ]:
# ========================================
# TEST SET EVALUATION - BEST RESNET18
# ========================================

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score
)

print("=======================================")
print("LOADING BEST 4-CLASS MODEL")
print("=======================================")

# ----------------------------------------
# Best model path
# ----------------------------------------

best_model_path = "/kaggle/working/best_resnet18_4class.pth"

checkpoint = torch.load(
    best_model_path,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

print("Best model loaded successfully")
print("Best validation accuracy:",
      checkpoint["val_accuracy"])


# ========================================
# TEST EVALUATION
# ========================================

print("\n=======================================")
print("EVALUATING TEST SET")
print("=======================================")

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Probabilities
        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        # Prediction
        _, predictions = torch.max(
            outputs,
            1
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )


# Convert to numpy
all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)


# ========================================
# ACCURACY
# ========================================

test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

balanced_acc = balanced_accuracy_score(
    all_labels,
    all_predictions
)

print("\n=======================================")
print("TEST RESULTS")
print("=======================================")

print(
    f"Test Accuracy: {test_accuracy * 100:.2f}%"
)

print(
    f"Balanced Accuracy: {balanced_acc * 100:.2f}%"
)


# ========================================
# CLASSIFICATION REPORT
# ========================================

print("\n=======================================")
print("CLASSIFICATION REPORT")
print("=======================================")

report = classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    zero_division=0
)

print(report)


# ========================================
# CONFUSION MATRIX
# ========================================

print("\n=======================================")
print("CONFUSION MATRIX")
print("=======================================")

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3]
)

print(cm)


# ========================================
# CONFUSION MATRIX WITH CLASS NAMES
# ========================================

print("\n=======================================")
print("CONFUSION MATRIX - LABELED")
print("=======================================")

cm_df = pd.DataFrame(
    cm,
    index=[
        f"Actual - {name}"
        for name in class_names
    ],
    columns=[
        f"Predicted - {name}"
        for name in class_names
    ]
)

print(cm_df)


# ========================================
# PER-CLASS CORRECT / TOTAL
# ========================================

print("\n=======================================")
print("PER-CLASS PERFORMANCE")
print("=======================================")

for i, class_name in enumerate(class_names):

    total_class = np.sum(
        all_labels == i
    )

    correct_class = np.sum(
        (all_labels == i) &
        (all_predictions == i)
    )

    if total_class > 0:
        class_acc = (
            100.0 *
            correct_class /
            total_class
        )
    else:
        class_acc = 0.0

    print(
        f"{class_name}: "
        f"{correct_class}/{total_class} "
        f"correct "
        f"({class_acc:.2f}%)"
    )


# ========================================
# FINAL
# ========================================

print("\n=======================================")
print("TEST EVALUATION COMPLETE")
print("=======================================")

print(
    f"Test Accuracy: "
    f"{test_accuracy * 100:.2f}%"
)

print(
    f"Balanced Accuracy: "
    f"{balanced_acc * 100:.2f}%"
)

print("=======================================")

In [ ]:
# ========================================
# FINAL MODEL PERFORMANCE VISUALIZATION
# ========================================

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

print("=======================================")
print("CREATING FINAL MODEL PERFORMANCE GRAPHS")
print("=======================================")


# ========================================
# 1. CONFUSION MATRIX
# ========================================

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3]
)

plt.figure(figsize=(9, 7))

plt.imshow(cm)

plt.title("Confusion Matrix - ResNet18 4-Class")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.xticks(
    range(4),
    class_names,
    rotation=25,
    ha="right"
)

plt.yticks(
    range(4),
    class_names
)

for i in range(4):
    for j in range(4):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()

plt.tight_layout()
plt.show()


# ========================================
# 2. PER-CLASS PRECISION / RECALL / F1
# ========================================

from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = (
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3],
        zero_division=0
    )
)

x = np.arange(len(class_names))
width = 0.25

plt.figure(figsize=(11, 6))

plt.bar(
    x - width,
    precision,
    width,
    label="Precision"
)

plt.bar(
    x,
    recall,
    width,
    label="Recall"
)

plt.bar(
    x + width,
    f1,
    width,
    label="F1-Score"
)

plt.xticks(
    x,
    class_names,
    rotation=20,
    ha="right"
)

plt.ylim(0, 1.05)

plt.ylabel("Score")
plt.title("Per-Class Performance")

plt.legend()

plt.tight_layout()
plt.show()


# ========================================
# 3. PRINT FINAL METRICS
# ========================================

print("\n=======================================")
print("FINAL CLASS PERFORMANCE")
print("=======================================")

for i, name in enumerate(class_names):

    print(f"\n{name}")

    print(
        f"Precision : {precision[i]:.4f}"
    )

    print(
        f"Recall    : {recall[i]:.4f}"
    )

    print(
        f"F1-Score  : {f1[i]:.4f}"
    )

    print(
        f"Support   : {support[i]}"
    )


# ========================================
# 4. OVERALL RESULTS
# ========================================

print("\n=======================================")
print("OVERALL RESULTS")
print("=======================================")

print(
    f"Test Accuracy      : "
    f"{test_accuracy * 100:.2f}%"
)

print(
    f"Balanced Accuracy  : "
    f"{balanced_acc * 100:.2f}%"
)

print(
    f"Best Validation Accuracy : "
    f"{checkpoint['val_accuracy']:.2f}%"
)

print("\n=======================================")
print("VISUALIZATION COMPLETE")
print("=======================================")

In [ ]:
# ========================================
# IMAGE + SYMPTOMS + CLINICAL RISK MODULE
# ========================================

import torch
import torch.nn.functional as F
from PIL import Image
import os

print("=======================================")
print("IMAGE + SYMPTOMS CLINICAL MODULE")
print("=======================================")


# ========================================
# CLASS MAPPING
# ========================================

class_names = [
    "Melanoma",
    "Basal Cell Carcinoma",
    "Squamous Cell Carcinoma",
    "Merkel Cell Carcinoma"
]


# ========================================
# IMAGE PREDICTION FUNCTION
# ========================================

def predict_cancer_type(image_path):

    if not os.path.exists(image_path):
        raise FileNotFoundError(
            f"Image not found: {image_path}"
        )

    image = Image.open(image_path).convert("RGB")

    # Use the same validation/test preprocessing
    image_tensor = test_transform(image).unsqueeze(0)
    image_tensor = image_tensor.to(device)

    model.eval()

    with torch.no_grad():

        outputs = model(image_tensor)

        probabilities = F.softmax(
            outputs,
            dim=1
        )

        confidence, prediction = torch.max(
            probabilities,
            dim=1
        )

    predicted_class = class_names[
        prediction.item()
    ]

    confidence_value = (
        confidence.item() * 100
    )

    probability_dict = {}

    for i, class_name in enumerate(class_names):

        probability_dict[class_name] = (
            probabilities[0, i].item() * 100
        )

    return (
        predicted_class,
        confidence_value,
        probability_dict
    )


# ========================================
# CLINICAL INFORMATION FUNCTION
# ========================================

def clinical_risk_assessment(
    lesion_size_mm=None,
    lesion_thickness_mm=None,
    ulceration=False,
    bleeding=False,
    rapid_growth=False,
    pain=False,
    lymph_node_swelling=False,
    distant_spread=False
):

    risk_flags = []

    # ------------------------------------
    # Lesion size
    # ------------------------------------

    if lesion_size_mm is not None:

        if lesion_size_mm > 20:

            risk_flags.append(
                "Lesion size > 20 mm"
            )


    # ------------------------------------
    # Tumor thickness
    # ------------------------------------

    if lesion_thickness_mm is not None:

        if lesion_thickness_mm > 4:

            risk_flags.append(
                "Tumor thickness > 4 mm"
            )


    # ------------------------------------
    # Ulceration
    # ------------------------------------

    if ulceration:

        risk_flags.append(
            "Ulceration reported"
        )


    # ------------------------------------
    # Bleeding
    # ------------------------------------

    if bleeding:

        risk_flags.append(
            "Bleeding reported"
        )


    # ------------------------------------
    # Rapid growth
    # ------------------------------------

    if rapid_growth:

        risk_flags.append(
            "Rapid growth reported"
        )


    # ------------------------------------
    # Pain
    # ------------------------------------

    if pain:

        risk_flags.append(
            "Pain reported"
        )


    # ------------------------------------
    # Lymph nodes
    # ------------------------------------

    if lymph_node_swelling:

        risk_flags.append(
            "Possible regional lymph-node involvement"
        )


    # ------------------------------------
    # Distant spread
    # ------------------------------------

    if distant_spread:

        risk_flags.append(
            "Possible distant spread"
        )


    # ====================================
    # RISK CATEGORY
    # ====================================

    if distant_spread:

        risk_category = "HIGH"

    elif lymph_node_swelling:

        risk_category = "HIGH"

    elif (
        lesion_thickness_mm is not None
        and lesion_thickness_mm > 4
    ):

        risk_category = "HIGH"

    elif (
        ulceration
        or rapid_growth
        or bleeding
    ):

        risk_category = "MODERATE"

    elif len(risk_flags) > 0:

        risk_category = "MODERATE"

    else:

        risk_category = "LOW"


    return risk_category, risk_flags


# ========================================
# STAGE INFORMATION CHECK
# ========================================

def stage_assessment(
    cancer_type,
    lesion_thickness_mm=None,
    ulceration=False,
    lymph_node_swelling=False,
    distant_spread=False
):

    # ------------------------------------
    # Important safety rule
    # ------------------------------------

    if distant_spread:

        return (
            "Possible advanced/metastatic disease",
            "Clinical imaging/pathology is required to confirm stage."
        )


    if lymph_node_swelling:

        return (
            "Possible regional lymph-node involvement",
            "Lymph-node examination/biopsy and clinical staging are required."
        )


    # ------------------------------------
    # Melanoma
    # ------------------------------------

    if cancer_type == "Melanoma":

        if lesion_thickness_mm is None:

            return (
                "Stage cannot be determined",
                "Melanoma staging requires tumor thickness and other clinical/pathologic findings."
            )

        if lesion_thickness_mm < 0.8:

            if ulceration:

                return (
                    "Possible early-stage melanoma",
                    "Ulceration changes melanoma T-category; formal staging is required."
                )

            else:

                return (
                    "Possible early-stage melanoma",
                    "Formal TNM staging is required."
                )

        elif lesion_thickness_mm <= 1.0:

            return (
                "Possible early-stage melanoma",
                "Formal TNM staging is required."
            )

        else:

            return (
                "Higher-risk primary melanoma feature",
                "Exact stage requires full TNM assessment."
            )


    # ------------------------------------
    # Merkel Cell Carcinoma
    # ------------------------------------

    elif cancer_type == "Merkel Cell Carcinoma":

        if lesion_size_mm is not None:

            if lesion_size_mm <= 20:

                return (
                    "Possible localized MCC",
                    "MCC staging requires lymph-node and metastasis assessment."
                )

            else:

                return (
                    "Higher-risk localized MCC feature",
                    "Formal TNM staging is required."
                )

        return (
            "Stage cannot be determined",
            "MCC staging requires tumor size plus nodal/metastatic assessment."
        )


    # ------------------------------------
    # BCC / SCC
    # ------------------------------------

    elif cancer_type in [
        "Basal Cell Carcinoma",
        "Squamous Cell Carcinoma"
    ]:

        return (
            "Stage cannot be determined from current inputs",
            "Formal staging depends on tumor site and clinical/pathologic findings."
        )


    return (
        "Stage cannot be determined",
        "Additional clinical information is required."
    )


# ========================================
# COMPLETE PREDICTION FUNCTION
# ========================================

def complete_prediction(
    image_path,
    lesion_size_mm=None,
    lesion_thickness_mm=None,
    ulceration=False,
    bleeding=False,
    rapid_growth=False,
    pain=False,
    lymph_node_swelling=False,
    distant_spread=False
):

    # ------------------------------------
    # Image prediction
    # ------------------------------------

    (
        cancer_type,
        confidence,
        probabilities
    ) = predict_cancer_type(
        image_path
    )


    # ------------------------------------
    # Clinical risk
    # ------------------------------------

    (
        risk_category,
        risk_flags
    ) = clinical_risk_assessment(
        lesion_size_mm=lesion_size_mm,
        lesion_thickness_mm=lesion_thickness_mm,
        ulceration=ulceration,
        bleeding=bleeding,
        rapid_growth=rapid_growth,
        pain=pain,
        lymph_node_swelling=lymph_node_swelling,
        distant_spread=distant_spread
    )


    # ------------------------------------
    # Stage assessment
    # ------------------------------------

    (
        stage_result,
        stage_note
    ) = stage_assessment(
        cancer_type=cancer_type,
        lesion_thickness_mm=lesion_thickness_mm,
        ulceration=ulceration,
        lymph_node_swelling=lymph_node_swelling,
        distant_spread=distant_spread
    )


    # ====================================
    # FINAL OUTPUT
    # ====================================

    print("\n")
    print("=======================================")
    print("FINAL PREDICTION")
    print("=======================================")

    print(
        f"Cancer type prediction : "
        f"{cancer_type}"
    )

    print(
        f"Image confidence       : "
        f"{confidence:.2f}%"
    )


    print("\n---------------------------------------")
    print("CLASS PROBABILITIES")
    print("---------------------------------------")

    for name, probability in probabilities.items():

        print(
            f"{name}: "
            f"{probability:.2f}%"
        )


    print("\n---------------------------------------")
    print("CLINICAL RISK")
    print("---------------------------------------")

    print(
        f"Risk category: "
        f"{risk_category}"
    )


    if len(risk_flags) > 0:

        print("\nRisk factors:")

        for flag in risk_flags:

            print(
                f"- {flag}"
            )

    else:

        print(
            "No entered risk flags."
        )


    print("\n---------------------------------------")
    print("STAGE ASSESSMENT")
    print("---------------------------------------")

    print(
        f"Result: {stage_result}"
    )

    print(
        f"Note: {stage_note}"
    )


    print("\n=======================================")
    print("PREDICTION COMPLETE")
    print("=======================================")


    return {
        "cancer_type": cancer_type,
        "confidence": confidence,
        "probabilities": probabilities,
        "risk_category": risk_category,
        "risk_flags": risk_flags,
        "stage_result": stage_result,
        "stage_note": stage_note
    }


print("\n=======================================")
print("CLINICAL MODULE READY")
print("=======================================")
print("Use complete_prediction(...) to make a prediction.")

In [ ]:
# ========================================
# CLINICAL RISK / STAGE ASSESSMENT
# ========================================

def assess_clinical_stage(
    lesion_size_mm=None,
    lesion_thickness_mm=None,
    ulceration=False,
    bleeding=False,
    rapid_growth=False,
    pain=False,
    lymph_node_swelling=False,
    distant_spread=False
):

    risk_flags = []

    # ------------------------------------
    # Check clinical features
    # ------------------------------------

    if lesion_size_mm is not None:
        if float(lesion_size_mm) > 20:
            risk_flags.append("Large lesion size")

    if lesion_thickness_mm is not None:
        if float(lesion_thickness_mm) > 4:
            risk_flags.append("High lesion thickness")

    if ulceration:
        risk_flags.append("Ulceration")

    if bleeding:
        risk_flags.append("Bleeding")

    if rapid_growth:
        risk_flags.append("Rapid growth")

    if pain:
        risk_flags.append("Pain")

    if lymph_node_swelling:
        risk_flags.append("Lymph node swelling")

    if distant_spread:
        risk_flags.append("Possible distant spread")


    # ------------------------------------
    # Risk / stage assessment
    # ------------------------------------

    if distant_spread:

        risk_category = "VERY HIGH"

        stage_assessment = (
            "Possible advanced disease / distant involvement"
        )

        note = (
            "Formal cancer staging requires appropriate "
            "clinical and pathological assessment."
        )

    elif lymph_node_swelling:

        risk_category = "HIGH"

        stage_assessment = (
            "Possible regional lymph-node involvement"
        )

        note = (
            "Lymph-node involvement requires clinical "
            "and pathological confirmation."
        )

    elif (
        (lesion_thickness_mm is not None and
         float(lesion_thickness_mm) > 4)
        or
        (lesion_size_mm is not None and
         float(lesion_size_mm) > 20)
        or
        ulceration
    ):

        risk_category = "HIGH"

        stage_assessment = (
            "Higher-risk clinical features detected"
        )

        note = (
            "The entered clinical features indicate higher risk, "
            "but they are not sufficient to determine a formal stage."
        )

    elif bleeding or rapid_growth or pain:

        risk_category = "MODERATE"

        stage_assessment = (
            "Moderate clinical risk features detected"
        )

        note = (
            "Symptoms indicate increased concern, but formal "
            "cancer staging cannot be determined from symptoms alone."
        )

    else:

        risk_category = "LOW"

        stage_assessment = (
            "No major high-risk clinical features entered"
        )

        note = (
            "This assessment is based only on the entered information "
            "and does not determine formal cancer stage."
        )


    # ------------------------------------
    # Return result
    # ------------------------------------

    return {
        "risk_category": risk_category,
        "stage_assessment": stage_assessment,
        "risk_flags": risk_flags,
        "note": note
    }


print("=======================================")
print("CLINICAL STAGE ASSESSMENT READY")
print("=======================================")

In [ ]:
# ========================================
# UPDATE COMPLETE PREDICTION WITH
# CLINICAL RISK + STAGE ASSESSMENT
# ========================================

def complete_prediction_with_stage(
    image_path,
    lesion_size_mm=None,
    lesion_thickness_mm=None,
    ulceration=False,
    bleeding=False,
    rapid_growth=False,
    pain=False,
    lymph_node_swelling=False,
    distant_spread=False
):

    # ------------------------------------
    # Cancer type prediction
    # ------------------------------------

    cancer_type, confidence, probabilities = predict_cancer_type(
        image_path
    )

    # ------------------------------------
    # Clinical assessment
    # ------------------------------------

    clinical_result = assess_clinical_stage(
        lesion_size_mm=lesion_size_mm,
        lesion_thickness_mm=lesion_thickness_mm,
        ulceration=ulceration,
        bleeding=bleeding,
        rapid_growth=rapid_growth,
        pain=pain,
        lymph_node_swelling=lymph_node_swelling,
        distant_spread=distant_spread
    )

    # ------------------------------------
    # Display result
    # ------------------------------------

    print("\n")
    print("=======================================")
    print("FINAL CLINICAL PREDICTION")
    print("=======================================")

    print(
        "Cancer type prediction :",
        cancer_type
    )

    print(
        "Image confidence       :",
        f"{confidence:.2f}%"
    )

    print("\n---------------------------------------")
    print("CLASS PROBABILITIES")
    print("---------------------------------------")

    for class_name, probability in probabilities.items():

        print(
            f"{class_name}: {probability:.2f}%"
        )

    print("\n---------------------------------------")
    print("CLINICAL RISK")
    print("---------------------------------------")

    print(
        "Risk category:",
        clinical_result["risk_category"]
    )

    if clinical_result["risk_flags"]:

        print("\nRisk flags:")

        for flag in clinical_result["risk_flags"]:
            print("-", flag)

    else:

        print("No entered risk flags.")


    print("\n---------------------------------------")
    print("STAGE ASSESSMENT")
    print("---------------------------------------")

    print(
        "Result:",
        clinical_result["stage_assessment"]
    )

    print(
        "Note:",
        clinical_result["note"]
    )

    print("\n=======================================")
    print("PREDICTION COMPLETE")
    print("=======================================")


    # ------------------------------------
    # Return all results
    # ------------------------------------

    return {
        "cancer_type": cancer_type,
        "confidence": confidence,
        "probabilities": probabilities,
        "risk_category": clinical_result["risk_category"],
        "risk_flags": clinical_result["risk_flags"],
        "stage_assessment": clinical_result["stage_assessment"],
        "note": clinical_result["note"]
    }


print("=======================================")
print("FINAL CLINICAL PREDICTION FUNCTION READY")
print("=======================================")

In [ ]:
# ========================================
# RESTORE TEST TRANSFORM
# ========================================

from torchvision import transforms

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("=======================================")
print("TEST TRANSFORM READY")
print("=======================================")

In [ ]:
# ========================================
# SET DEVICE
# ========================================

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=======================================")
print("DEVICE READY")
print("=======================================")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

In [ ]:
# ========================================
# CLASS MAPPING
# ========================================

class_names = [
    "Melanoma",
    "Basal Cell Carcinoma",
    "Squamous Cell Carcinoma",
    "Merkel Cell Carcinoma"
]

print("=======================================")
print("CLASS NAMES READY")
print("=======================================")

for i, name in enumerate(class_names):
    print(f"{i} = {name}")

In [ ]:
# ========================================
# IMAGE PREDICTION FUNCTION
# ========================================

import os
import torch
import torch.nn.functional as F
from PIL import Image

def predict_cancer_type(image_path):

    if not os.path.exists(image_path):
        raise FileNotFoundError(
            f"Image not found: {image_path}"
        )

    image = Image.open(image_path).convert("RGB")

    # Same preprocessing used for validation/test
    image_tensor = test_transform(image).unsqueeze(0)
    image_tensor = image_tensor.to(device)

    model.eval()

    with torch.no_grad():

        outputs = model(image_tensor)

        probabilities = F.softmax(
            outputs,
            dim=1
        )

        confidence, prediction = torch.max(
            probabilities,
            dim=1
        )

    predicted_class = class_names[
        prediction.item()
    ]

    confidence_value = (
        confidence.item() * 100
    )

    probability_dict = {}

    for i, class_name in enumerate(class_names):

        probability_dict[class_name] = (
            probabilities[0, i].item() * 100
        )

    return (
        predicted_class,
        confidence_value,
        probability_dict
    )


print("=======================================")
print("IMAGE PREDICTION FUNCTION READY")
print("=======================================")

In [ ]:
# =======================================
# CLINICAL FEATURE SCORES
# =======================================

print("=======================================")
print("CLINICAL FEATURE SCORES READY")
print("=======================================")

# Default values
# Clinical feature input module was removed,
# so validation can run without those inputs.

melanoma_score = 0
bcc_score = 0
scc_score = 0
mcc_score = 0

print("Melanoma ABCDE :", melanoma_score, "/5")
print("BCC features   :", bcc_score, "/4")
print("SCC features   :", scc_score, "/4")
print("MCC AEIOU      :", mcc_score, "/5")

In [ ]:
# ======================================= 
# CLINICAL VALIDATION VARIABLES 
# ======================================= 
 
clinical_risk = "INSUFFICIENT INFORMATION" 
risk_flags = [] 
 
print("=======================================") 
print("CLINICAL VALIDATION VARIABLES READY") 
print("=======================================") 
print("Clinical risk:", clinical_risk) 
print("Risk flags:", risk_flags)

In [ ]:
# =======================================
# FINAL SYSTEM VALIDATION - MULTIPLE IMAGES
# =======================================

print("\n")
print("=======================================")
print("FINAL SYSTEM VALIDATION")
print("=======================================")

# =======================================
# TEST IMAGES
# =======================================

validation_images = [

    "/kaggle/input/datasets/tomooinubushi/"
    "all-isic-data-20240629/images/ISIC_0066441.jpg",

    # Add more test-image paths below
    # Example:
    # "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/images/ISIC_XXXXXXX.jpg",
]


# =======================================
# VALIDATE IMAGES
# =======================================

validation_results = []


for i, image_path in enumerate(validation_images):

    print("\n")
    print("=======================================")
    print(f"TEST IMAGE {i + 1}")
    print("=======================================")

    print("Image:", image_path)

    try:

        # -----------------------------------
        # Image prediction
        # -----------------------------------

        image_result = predict_cancer_type(
            image_path
        )

        image_cancer_type = image_result[0]
        image_confidence = image_result[1]
        image_probabilities = image_result[2]


        # -----------------------------------
        # Clinical scores
        # -----------------------------------

        clinical_scores_validation = {

            "Melanoma":
                melanoma_score / 5.0,

            "Basal Cell Carcinoma":
                bcc_score / 4.0,

            "Squamous Cell Carcinoma":
                scc_score / 4.0,

            "Merkel Cell Carcinoma":
                mcc_score / 5.0
        }


        # -----------------------------------
        # Combined prediction
        # -----------------------------------

        validation_combined_scores = {}

        for class_name in class_names:

            img_score = (
                float(
                    image_probabilities.get(
                        class_name,
                        0
                    )
                ) / 100.0
            )

            clinical_score = (
                clinical_scores_validation[
                    class_name
                ]
            )

            validation_combined_scores[
                class_name
            ] = (
                0.70 * img_score
                +
                0.30 * clinical_score
            )


        # -----------------------------------
        # Normalize
        # -----------------------------------

        total_score = sum(
            validation_combined_scores.values()
        )


        validation_combined_probabilities = {

            class_name:
                (
                    validation_combined_scores[
                        class_name
                    ] / total_score
                ) * 100.0

            for class_name in class_names
        }


        # -----------------------------------
        # Final prediction
        # -----------------------------------

        final_prediction = max(
            validation_combined_probabilities,
            key=validation_combined_probabilities.get
        )

        final_confidence = (
            validation_combined_probabilities[
                final_prediction
            ]
        )


        # -----------------------------------
        # Display
        # -----------------------------------

        print("\nImage prediction:")
        print(
            image_cancer_type,
            f"({image_confidence:.2f}%)"
        )

        print("\nCombined prediction:")
        print(
            final_prediction,
            f"({final_confidence:.2f}%)"
        )

        print("\nClinical support:")

        if final_prediction == "Melanoma":

            support_score = melanoma_score
            support_total = 5

        elif final_prediction == "Basal Cell Carcinoma":

            support_score = bcc_score
            support_total = 4

        elif final_prediction == "Squamous Cell Carcinoma":

            support_score = scc_score
            support_total = 4

        else:

            support_score = mcc_score
            support_total = 5


        support_ratio = (
            support_score / support_total
        )


        if support_ratio >= 0.60:

            support = "STRONG"

        elif support_ratio >= 0.30:

            support = "MODERATE"

        else:

            support = "LIMITED"


        print(support)


        # -----------------------------------
        # Save result
        # -----------------------------------

        validation_results.append({

            "image":
                image_path,

            "image_prediction":
                image_cancer_type,

            "image_confidence":
                round(
                    float(image_confidence),
                    2
                ),

            "final_prediction":
                final_prediction,

            "final_confidence":
                round(
                    float(final_confidence),
                    2
                ),

            "clinical_support":
                support,

            "clinical_risk":
                clinical_risk,

            "clinical_stage":
                clinical_stage
        })


    except Exception as e:

        print("\nERROR:")
        print(e)


# =======================================
# FINAL VALIDATION TABLE
# =======================================

print("\n")
print("=======================================")
print("VALIDATION RESULTS")
print("=======================================")


if validation_results:

    validation_df = pd.DataFrame(
        validation_results
    )

    print(
        validation_df.to_string(
            index=False
        )
    )

else:

    print(
        "No validation results generated."
    )


print("\n")
print("=======================================")
print("FINAL SYSTEM VALIDATION COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# MULTIPLE TEST IMAGE VALIDATION
# =======================================

print("\n")
print("=======================================")
print("MULTIPLE TEST IMAGE VALIDATION")
print("=======================================")


# =======================================
# ADD TEST IMAGE PATHS HERE
# =======================================

validation_images = [

    "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/images/ISIC_0066441.jpg",

    # Add more test images here
    # Example:
    # "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/images/ISIC_XXXXXXX.jpg",
    # "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/images/ISIC_XXXXXXX.jpg",

]


# =======================================
# VALIDATION RESULTS
# =======================================

all_validation_results = []


# =======================================
# PROCESS EACH IMAGE
# =======================================

for i, image_path in enumerate(validation_images):

    print("\n")
    print("=======================================")
    print(f"TEST IMAGE {i + 1}")
    print("=======================================")

    print("Image:", image_path)

    try:

        # -----------------------------------
        # IMAGE MODEL PREDICTION
        # -----------------------------------

        result = predict_cancer_type(
            image_path
        )

        image_prediction = result[0]
        image_confidence = result[1]
        image_probabilities = result[2]


        # -----------------------------------
        # CLINICAL SCORES
        # -----------------------------------

        clinical_scores = {

            "Melanoma":
                melanoma_score / 5.0,

            "Basal Cell Carcinoma":
                bcc_score / 4.0,

            "Squamous Cell Carcinoma":
                scc_score / 4.0,

            "Merkel Cell Carcinoma":
                mcc_score / 5.0
        }


        # -----------------------------------
        # COMBINE IMAGE + CLINICAL
        # -----------------------------------

        combined_scores = {}

        for class_name in class_names:

            image_score = (
                float(
                    image_probabilities.get(
                        class_name,
                        0.0
                    )
                ) / 100.0
            )

            clinical_score = (
                clinical_scores[class_name]
            )

            combined_scores[class_name] = (
                0.70 * image_score
                +
                0.30 * clinical_score
            )


        # -----------------------------------
        # NORMALIZE
        # -----------------------------------

        total = sum(
            combined_scores.values()
        )

        combined_probabilities = {

            class_name:
                (
                    combined_scores[class_name]
                    / total
                ) * 100.0

            for class_name in class_names
        }


        # -----------------------------------
        # FINAL PREDICTION
        # -----------------------------------

        final_prediction = max(
            combined_probabilities,
            key=combined_probabilities.get
        )

        final_confidence = (
            combined_probabilities[
                final_prediction
            ]
        )


        # -----------------------------------
        # CLINICAL SUPPORT
        # -----------------------------------

        if final_prediction == "Melanoma":

            score = melanoma_score
            maximum = 5

        elif final_prediction == "Basal Cell Carcinoma":

            score = bcc_score
            maximum = 4

        elif final_prediction == "Squamous Cell Carcinoma":

            score = scc_score
            maximum = 4

        else:

            score = mcc_score
            maximum = 5


        support_ratio = score / maximum


        if support_ratio >= 0.60:

            support = "STRONG"

        elif support_ratio >= 0.30:

            support = "MODERATE"

        else:

            support = "LIMITED"


        # -----------------------------------
        # DISPLAY RESULT
        # -----------------------------------

        print("\nImage prediction:")
        print(
            f"{image_prediction} "
            f"({image_confidence:.2f}%)"
        )

        print("\nCombined prediction:")
        print(
            f"{final_prediction} "
            f"({final_confidence:.2f}%)"
        )

        print(
            "Clinical support:",
            support
        )

        print(
            "Clinical risk:",
            clinical_risk
        )

        print(
            "Clinical stage:",
            clinical_stage
        )


        # -----------------------------------
        # SAVE RESULT
        # -----------------------------------

        all_validation_results.append({

            "Image":
                image_path,

            "Image Prediction":
                image_prediction,

            "Image Confidence":
                round(
                    float(image_confidence),
                    2
                ),

            "Combined Prediction":
                final_prediction,

            "Combined Confidence":
                round(
                    float(final_confidence),
                    2
                ),

            "Clinical Support":
                support,

            "Clinical Risk":
                clinical_risk,

            "Clinical Stage":
                clinical_stage
        })


    except Exception as e:

        print("\nERROR:")
        print(e)


# =======================================
# CREATE VALIDATION TABLE
# =======================================

print("\n")
print("=======================================")
print("FINAL VALIDATION TABLE")
print("=======================================")


if len(all_validation_results) > 0:

    final_validation_df = pd.DataFrame(
        all_validation_results
    )

    print(
        final_validation_df.to_string(
            index=False
        )
    )

else:

    print(
        "No validation results available."
    )


# =======================================
# VALIDATION COMPLETE
# =======================================

print("\n")
print("=======================================")
print("MULTIPLE IMAGE VALIDATION COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# CHECK AVAILABLE TEST IMAGES
# =======================================

print("=======================================")
print("FINDING TEST IMAGES")
print("=======================================")

# Get test image paths from the actual test dataframe
available_test_images = test_df[
    "image_path"
].dropna().drop_duplicates().tolist()

print(
    "Available test images:",
    len(available_test_images)
)

print("\nFirst 20 test images:")
print("---------------------------------------")

for i, path in enumerate(
    available_test_images[:20]
):

    print(
        f"{i + 1}: {path}"
    )

print("\n")
print("=======================================")
print("TEST IMAGE LIST READY")
print("=======================================")

In [ ]:
# =======================================
# FINAL 1292-IMAGE TEST SET EVALUATION
# =======================================

import torch
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

print("=======================================")
print("FINAL TEST SET EVALUATION")
print("=======================================")

# ---------------------------------------
# Load best trained model
# ---------------------------------------

checkpoint = torch.load(
    best_model_path,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

print("Best model loaded successfully")
print(
    "Best validation accuracy:",
    checkpoint["val_accuracy"]
)

# ---------------------------------------
# Prediction on complete test set
# ---------------------------------------

all_predictions = []
all_labels = []
all_probabilities = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )

# ---------------------------------------
# Convert to numpy
# ---------------------------------------

y_true = np.array(all_labels)
y_pred = np.array(all_predictions)
y_prob = np.array(all_probabilities)

# ---------------------------------------
# Accuracy
# ---------------------------------------

test_accuracy = accuracy_score(
    y_true,
    y_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred
)

print("\n=======================================")
print("TEST SET RESULTS")
print("=======================================")

print(
    f"Test Accuracy: {test_accuracy * 100:.2f}%"
)

print(
    f"Balanced Accuracy: {balanced_accuracy * 100:.2f}%"
)

# ---------------------------------------
# Classification report
# ---------------------------------------

print("\n=======================================")
print("CLASSIFICATION REPORT")
print("=======================================")

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

# ---------------------------------------
# Confusion matrix
# ---------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3]
)

print("\n=======================================")
print("CONFUSION MATRIX")
print("=======================================")

print(
    pd.DataFrame(
        cm,
        index=[
            f"Actual - {name}"
            for name in class_names
        ],
        columns=[
            f"Predicted - {name}"
            for name in class_names
        ]
    )
)

# ---------------------------------------
# Per-class performance
# ---------------------------------------

print("\n=======================================")
print("PER-CLASS PERFORMANCE")
print("=======================================")

for i, class_name in enumerate(class_names):

    class_total = cm[i].sum()
    class_correct = cm[i, i]

    if class_total > 0:
        class_recall = (
            class_correct / class_total
        ) * 100
    else:
        class_recall = 0.0

    print(
        f"{class_name}: "
        f"{class_correct}/{class_total} correct "
        f"({class_recall:.2f}%)"
    )

# ---------------------------------------
# Save predictions
# ---------------------------------------

test_results = test_df.copy().reset_index(
    drop=True
)

test_results["true_label"] = y_true
test_results["predicted_label"] = y_pred

test_results["true_class"] = (
    test_results["true_label"]
    .map(dict(enumerate(class_names)))
)

test_results["predicted_class"] = (
    test_results["predicted_label"]
    .map(dict(enumerate(class_names)))
)

test_results["prediction_confidence"] = (
    np.max(y_prob, axis=1) * 100
)

# ---------------------------------------
# Save CSV
# ---------------------------------------

test_results_path = (
    "/kaggle/working/"
    "final_1292_test_predictions.csv"
)

test_results.to_csv(
    test_results_path,
    index=False
)

print("\n=======================================")
print("TEST PREDICTIONS SAVED")
print("=======================================")

print(test_results_path)

print("\n=======================================")
print("FINAL TEST EVALUATION COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# CANCER-SPECIFIC CLINICAL FEATURE INPUT
# ABCDE + BCC + SCC + AEIOU
# =======================================

print("=======================================")
print("CANCER-SPECIFIC CLINICAL ASSESSMENT")
print("=======================================")

print("""
This module combines the image prediction with
cancer-specific clinical features.

Melanoma -> ABCDE
BCC      -> Nodular BCC clinical features
SCC      -> Clinical features
MCC      -> AEIOU
""")

# =======================================
# HELPER
# =======================================

def yes_no_to_bool(value):
    return str(value).strip().lower() in [
        "yes", "y", "true", "1"
    ]


# =======================================
# MELANOMA - ABCDE
# =======================================

print("\n=======================================")
print("1. MELANOMA - ABCDE RULE")
print("=======================================")

A = input(
    "A - Asymmetry present? (yes/no): "
)

B = input(
    "B - Border irregularity present? (yes/no): "
)

C = input(
    "C - Color variation present? (yes/no): "
)

diameter_input = input(
    "D - Diameter > 6 mm? (yes/no): "
)

E = input(
    "E - Evolving lesion (size/color/shape change)? (yes/no): "
)

melanoma_features = {
    "Asymmetry": yes_no_to_bool(A),
    "Border irregularity": yes_no_to_bool(B),
    "Color variation": yes_no_to_bool(C),
    "Diameter > 6 mm": yes_no_to_bool(diameter_input),
    "Evolving": yes_no_to_bool(E)
}

melanoma_score = sum(melanoma_features.values())


# =======================================
# BCC CLINICAL FEATURES
# =======================================

print("\n=======================================")
print("2. BASAL CELL CARCINOMA - CLINICAL FEATURES")
print("=======================================")

bcc_pearly = input(
    "Pearly/shiny bump present? (yes/no): "
)

bcc_rolled = input(
    "Rolled border present? (yes/no): "
)

bcc_telangiectasia = input(
    "Small visible blood vessels/telangiectasia? (yes/no): "
)

bcc_ulcer = input(
    "Center ulcer present? (yes/no): "
)

bcc_features = {
    "Pearly/shiny bump": yes_no_to_bool(bcc_pearly),
    "Rolled border": yes_no_to_bool(bcc_rolled),
    "Telangiectasia": yes_no_to_bool(bcc_telangiectasia),
    "Center ulcer": yes_no_to_bool(bcc_ulcer)
}

bcc_score = sum(bcc_features.values())


# =======================================
# SCC CLINICAL FEATURES
# =======================================

print("\n=======================================")
print("3. SQUAMOUS CELL CARCINOMA - CLINICAL FEATURES")
print("=======================================")

scc_nodule = input(
    "Firm red nodule present? (yes/no): "
)

scc_scaly = input(
    "Scaly/crusted plaque present? (yes/no): "
)

scc_raised_ulcer = input(
    "Raised ulcer edge present? (yes/no): "
)

scc_keratin = input(
    "Thick keratin layer present? (yes/no): "
)

scc_features = {
    "Firm red nodule": yes_no_to_bool(scc_nodule),
    "Scaly/crusted plaque": yes_no_to_bool(scc_scaly),
    "Raised ulcer edge": yes_no_to_bool(scc_raised_ulcer),
    "Thick keratin layer": yes_no_to_bool(scc_keratin)
}

scc_score = sum(scc_features.values())


# =======================================
# MCC - AEIOU
# =======================================

print("\n=======================================")
print("4. MERKEL CELL CARCINOMA - AEIOU RULE")
print("=======================================")

mcc_A = input(
    "A - Asymptomatic/painless lesion? (yes/no): "
)

mcc_E = input(
    "E - Expanding rapidly? (yes/no): "
)

mcc_I = input(
    "I - Immunosuppressed patient? (yes/no): "
)

mcc_O = input(
    "O - Patient older than 50? (yes/no): "
)

mcc_U = input(
    "U - UV-exposed skin? (yes/no): "
)

mcc_features = {
    "Asymptomatic": yes_no_to_bool(mcc_A),
    "Expanding rapidly": yes_no_to_bool(mcc_E),
    "Immunosuppressed": yes_no_to_bool(mcc_I),
    "Older than 50": yes_no_to_bool(mcc_O),
    "UV exposed skin": yes_no_to_bool(mcc_U)
}

mcc_score = sum(mcc_features.values())


# =======================================
# FEATURE SUMMARY
# =======================================

print("\n=======================================")
print("CLINICAL FEATURE SUMMARY")
print("=======================================")

print(
    f"Melanoma ABCDE score : {melanoma_score}/5"
)

print(
    f"BCC clinical score   : {bcc_score}/4"
)

print(
    f"SCC clinical score   : {scc_score}/4"
)

print(
    f"MCC AEIOU score      : {mcc_score}/5"
)


# =======================================
# DISPLAY POSITIVE FEATURES
# =======================================

print("\n=======================================")
print("POSITIVE CLINICAL FEATURES")
print("=======================================")

print("\nMelanoma / ABCDE:")
for feature, present in melanoma_features.items():
    if present:
        print("✓", feature)

print("\nBCC:")
for feature, present in bcc_features.items():
    if present:
        print("✓", feature)

print("\nSCC:")
for feature, present in scc_features.items():
    if present:
        print("✓", feature)

print("\nMCC / AEIOU:")
for feature, present in mcc_features.items():
    if present:
        print("✓", feature)


# =======================================
# CLINICAL SUPPORT SCORES
# =======================================

clinical_scores = {
    "Melanoma": melanoma_score / 5.0,
    "Basal Cell Carcinoma": bcc_score / 4.0,
    "Squamous Cell Carcinoma": scc_score / 4.0,
    "Merkel Cell Carcinoma": mcc_score / 5.0
}


# =======================================
# IMAGE PREDICTION
# =======================================

print("\n=======================================")
print("IMAGE MODEL PREDICTION")
print("=======================================")

image_result = predict_cancer_type(test_image_path)

image_cancer = image_result[0]
image_confidence = image_result[1]
image_probabilities = image_result[2]

print(
    "Image prediction:",
    image_cancer
)

print(
    f"Image confidence: {image_confidence:.2f}%"
)


# =======================================
# COMBINE IMAGE + CLINICAL INFORMATION
# =======================================

# Image probabilities are used as the main model evidence.
# Clinical features are used as additional supporting evidence.

combined_scores = {}

for class_name in [
    "Melanoma",
    "Basal Cell Carcinoma",
    "Squamous Cell Carcinoma",
    "Merkel Cell Carcinoma"
]:

    image_score = (
        image_probabilities[class_name] / 100.0
    )

    clinical_score = clinical_scores[class_name]

    # 80% image + 20% clinical support
    combined_scores[class_name] = (
        0.80 * image_score +
        0.20 * clinical_score
    )


# =======================================
# FINAL TYPE
# =======================================

final_cancer_type = max(
    combined_scores,
    key=combined_scores.get
)

final_score = combined_scores[
    final_cancer_type
] * 100


# =======================================
# CLINICAL SUPPORT LEVEL
# =======================================

support_score = clinical_scores[
    final_cancer_type
]

if support_score >= 0.75:
    clinical_support = "STRONG"
elif support_score >= 0.50:
    clinical_support = "MODERATE"
elif support_score >= 0.25:
    clinical_support = "WEAK"
else:
    clinical_support = "LOW"


# =======================================
# FINAL RESULT
# =======================================

print("\n")
print("=======================================")
print("FINAL IMAGE + CLINICAL PREDICTION")
print("=======================================")

print(
    "Cancer type prediction:",
    final_cancer_type
)

print(
    f"Combined prediction score: {final_score:.2f}%"
)

print(
    "Clinical feature support:",
    clinical_support
)

print("\n---------------------------------------")
print("IMAGE MODEL PROBABILITIES")
print("---------------------------------------")

for class_name, probability in image_probabilities.items():

    print(
        f"{class_name}: {probability:.2f}%"
    )


print("\n---------------------------------------")
print("CLINICAL FEATURE SCORES")
print("---------------------------------------")

print(
    f"Melanoma ABCDE: "
    f"{melanoma_score}/5"
)

print(
    f"BCC features: "
    f"{bcc_score}/4"
)

print(
    f"SCC features: "
    f"{scc_score}/4"
)

print(
    f"MCC AEIOU: "
    f"{mcc_score}/5"
)


# =======================================
# SAFETY / CLINICAL NOTE
# =======================================

print("\n=======================================")
print("CLINICAL NOTE")
print("=======================================")

print(
    "ABCDE, BCC, SCC and AEIOU findings are "
    "clinical support features."
)

print(
    "They do not independently establish a "
    "definitive cancer diagnosis."
)

print(
    "Formal cancer staging requires appropriate "
    "clinical and pathological information."
)

print("\n=======================================")
print("ASSESSMENT COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# CLINICAL STAGE CATEGORY
# EARLY / INTERMEDIATE / ADVANCED
# =======================================

print("=======================================")
print("CLINICAL STAGE CATEGORY ASSESSMENT")
print("=======================================")

print("""
This is a clinical risk-based category.
It is NOT a formal cancer stage.

Please enter only known/confirmed information.
Use 'unknown' when the information is not available.
""")

# =======================================
# STAGE-RELEVANT INFORMATION
# =======================================

tumor_size_input = input(
    "Known tumor size in mm? (enter 0 if unknown): "
).strip()

tumor_thickness_input = input(
    "Known tumor thickness in mm? (enter 0 if unknown): "
).strip()

ulceration_stage = input(
    "Ulceration confirmed? (yes/no/unknown): "
).strip().lower()

lymph_stage = input(
    "Regional lymph-node involvement confirmed? (yes/no/unknown): "
).strip().lower()

metastasis_stage = input(
    "Distant metastasis confirmed? (yes/no/unknown): "
).strip().lower()


# =======================================
# SAFE NUMERIC CONVERSION
# =======================================

try:
    tumor_size_mm = float(tumor_size_input)
except:
    tumor_size_mm = 0.0

try:
    tumor_thickness_mm = float(tumor_thickness_input)
except:
    tumor_thickness_mm = 0.0


# =======================================
# CLINICAL STAGE CATEGORY
# =======================================

if metastasis_stage in ["yes", "y", "true", "1"]:

    clinical_stage_category = "ADVANCED"
    stage_reason = "Confirmed distant metastatic disease."

elif lymph_stage in ["yes", "y", "true", "1"]:

    clinical_stage_category = "ADVANCED"
    stage_reason = "Confirmed regional lymph-node involvement."

elif (
    ulceration_stage in ["yes", "y", "true", "1"]
    or tumor_thickness_mm > 0
    or tumor_size_mm > 0
):

    clinical_stage_category = "INTERMEDIATE"
    stage_reason = (
        "Tumor/stage-relevant clinical features are present, "
        "but no confirmed regional lymph-node involvement "
        "or distant metastasis was entered."
    )

else:

    clinical_stage_category = "EARLY"
    stage_reason = (
        "No confirmed regional lymph-node involvement, "
        "distant metastasis, or other major stage-related "
        "clinical findings were entered."
    )


# =======================================
# DISPLAY RESULT
# =======================================

print("\n=======================================")
print("CLINICAL STAGE CATEGORY")
print("=======================================")

print(
    "Clinical stage category:",
    clinical_stage_category
)

print("\nReason:")
print(stage_reason)


# =======================================
# AVAILABLE INFORMATION
# =======================================

print("\n=======================================")
print("STAGE-RELEVANT INFORMATION")
print("=======================================")

print(f"Known tumor size: {tumor_size_mm:.1f} mm")
print(f"Known tumor thickness: {tumor_thickness_mm:.1f} mm")
print("Ulceration:", ulceration_stage)
print("Regional lymph-node involvement:", lymph_stage)
print("Distant metastasis:", metastasis_stage)


# =======================================
# INTERPRETATION
# =======================================

print("\n=======================================")
print("CLINICAL STAGE INTERPRETATION")
print("=======================================")

if clinical_stage_category == "EARLY":

    print(
        "EARLY: No major advanced clinical findings "
        "were entered."
    )

elif clinical_stage_category == "INTERMEDIATE":

    print(
        "INTERMEDIATE: Stage-relevant clinical features "
        "are present, but no confirmed regional lymph-node "
        "involvement or distant metastasis was entered."
    )

else:

    print(
        "ADVANCED: Confirmed regional lymph-node involvement "
        "or distant metastatic disease was entered."
    )


# =======================================
# IMPORTANT NOTE
# =======================================

print("\n=======================================")
print("IMPORTANT CLINICAL NOTE")
print("=======================================")

print(
    "Early/Intermediate/Advanced is a clinical "
    "risk-based category created from the entered "
    "information."
)

print(
    "It is NOT a formal cancer stage and does not "
    "replace cancer-specific TNM/staging assessment."
)

print(
    "Formal diagnosis and staging require appropriate "
    "clinical and pathological evaluation."
)

print("\n=======================================")
print("CLINICAL STAGE CATEGORY COMPLETE")
print("=======================================")

In [ ]:
# =======================================
# CHECK DUPLICATE IMAGES / DATA LEAKAGE
# =======================================

print("=======================================")
print("DUPLICATE IMAGE CHECK")
print("=======================================")

# Duplicate image paths
duplicate_paths = combined_df[
    combined_df.duplicated(
        subset=["image_path"],
        keep=False
    )
]

print("Total rows:", len(combined_df))
print("Unique image paths:", combined_df["image_path"].nunique())
print("Duplicate image paths:", len(duplicate_paths))

# Duplicate image names
duplicate_names = combined_df[
    combined_df.duplicated(
        subset=["image_name"],
        keep=False
    )
]

print("\nDuplicate image names:", len(duplicate_names))

# Check each class
print("\n=======================================")
print("CLASS COUNTS")
print("=======================================")

for label, name in {
    0: "Melanoma",
    1: "Basal Cell Carcinoma",
    2: "Squamous Cell Carcinoma",
    3: "Merkel Cell Carcinoma"
}.items():

    count = (combined_df["label"] == label).sum()
    print(f"{label} = {name}: {count}")

In [ ]:
# =======================================
# CREATE PYTORCH DATASETS + DATALOADERS
# =======================================

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

# =======================================
# IMAGE TRANSFORMS
# =======================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


# =======================================
# CUSTOM DATASET
# =======================================

class SkinCancerDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["image_path"]
        label = int(row["label"])

        # Open image
        image = Image.open(image_path).convert("RGB")

        # Apply transform
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


# =======================================
# CREATE DATASETS
# =======================================

train_dataset = SkinCancerDataset(
    train_df,
    transform=train_transform
)

val_dataset = SkinCancerDataset(
    val_df,
    transform=test_transform
)

test_dataset = SkinCancerDataset(
    test_df,
    transform=test_transform
)


# =======================================
# CREATE DATALOADERS
# =======================================

batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# =======================================
# RESULTS
# =======================================

print("=======================================")
print("DATALOADERS CREATED")
print("=======================================")

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

print("Batch size:", batch_size)

In [ ]:
# =======================================
# CREATE 4-CLASS RESNET18 MODEL
# =======================================

import torch
import torch.nn as nn
from torchvision import models

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# Load pretrained ResNet18
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Replace final layer
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    4
)

# Move model to device
model = model.to(device)

print("=======================================")
print("4-CLASS MODEL CREATED")
print("=======================================")

print(model.fc)

print("\nClass mapping:")
print("0 = Melanoma")
print("1 = Basal Cell Carcinoma")
print("2 = Squamous Cell Carcinoma")
print("3 = Merkel Cell Carcinoma")

In [ ]:
# ========================================
# CHECK GPU / CUDA
# ========================================

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

    try:
        x = torch.randn(100, 100, device="cuda")
        y = x @ x
        print("GPU TEST: SUCCESS")
    except Exception as e:
        print("GPU TEST: FAILED")
        print(e)
else:
    print("GPU is NOT available")

In [ ]:
# ========================================
# FIX CUDA FOR TESLA P100
# ========================================

import subprocess
import sys

print("Installing PyTorch version compatible with Tesla P100...")

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--force-reinstall",
    "torch==2.5.1",
    "torchvision==0.20.1",
    "torchaudio==2.5.1",
    "--index-url",
    "https://download.pytorch.org/whl/cu118"
])

print("\n=======================================")
print("INSTALLATION COMPLETE")
print("=======================================")
print("IMPORTANT: Restart the Kaggle session/kernel now.")

In [ ]:
# ========================================
# CHECK PYTORCH / GPU
# ========================================

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))

In [ ]:
# ========================================
# FIX PYTORCH FOR TESLA P100
# ========================================

!pip uninstall -y torch torchvision torchaudio

!pip install -q torch==2.5.1 torchvision==0.20.1

In [ ]:
for root,dirs,files in os.walk("/kaggle/input/datasets/hiro002/dermacon-in-dataset"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root,file))

In [ ]:
derma=pd.read_csv(
"/kaggle/input/datasets/hiro002/dermacon-in-dataset/METADATA/Skin_Metadata.csv"
)
print(derma.head())

In [ ]:
print("HAM")
print(ham.columns)

print("\nISIC")
print(isic.columns)

print("\nDERMA")
print(derma.columns)

In [ ]:
ham_new=pd.DataFrame()

ham_new["image_name"]=ham["image_id"]

ham_new["class_name"]=ham["dx"]

ham_new["age"]=ham["age"]

ham_new["sex"]=ham["sex"]

ham_new["dataset"]="HAM10000"
ham_mapping = {
    "mel":0,
    "bcc":1,
    "scc":2
}

ham_new["label"] = ham_new["class_name"].map(ham_mapping)

ham_new = ham_new.dropna(subset=["label"])

In [ ]:
isic_cancer = isic[
    isic["diagnosis"].str.contains(
        "melanoma|basal cell carcinoma|squamous cell carcinoma",
        case=False,
        na=False
    )
].copy()

In [ ]:
isic_new=pd.DataFrame()

isic_new["image_name"]=isic["isic_id"]

isic_new["class_name"]=isic["diagnosis"]

isic_new["age"]=isic["age_approx"]

isic_new["sex"]=isic["sex"]

isic_new["dataset"]="isic2024"

In [ ]:
isic_mapping = {

    "melanoma":0,

    "melanoma metastasis":0,

    "basal cell carcinoma":1,

    "squamous cell carcinoma":2

}

isic_new["label"] = (
    isic_new["class_name"]
    .str.lower()
    .map(isic_mapping)
)

isic_new = isic_new.dropna(subset=["label"])

In [ ]:
derma_cancer = derma[
    derma["Disease_label"].str.contains(
        "Melanoma|Basal Cell Carcinoma|Squamous Cell Carcinoma",
        case=False,
        na=False
    )
].copy()

In [ ]:
derma_new=pd.DataFrame()

derma_new["image_name"]=derma["Image_name"]

derma_new["class_name"]=derma["Disease_label"]

derma_new["age"]=derma["Age"]

derma_new["sex"]=derma["Sex"]

derma_new["dataset"]="dermacon_in"

In [ ]:
derma_mapping = {

    "Melanoma":0,

    "Basal Cell Carcinoma":1,

    "Squamous Cell Carcinoma":2

}

derma_new["label"] = derma_new["class_name"].map(derma_mapping)

derma_new = derma_new.dropna(subset=["label"])

In [ ]:
print(ham_new.shape)

print(isic_new.shape)

print(derma_new.shape)

In [ ]:
combined = pd.concat(
    [ham_new, isic_new, derma_new],
    ignore_index=True
)

In [ ]:
print(combined["dataset"].value_counts())

print(combined["label"].value_counts())

In [ ]:
# Dataset Shape
print("Dataset Shape:", combined.shape)

# Column Names
print("\nColumns:")
print(combined.columns.tolist())

# First Five Records
display(combined.head())

In [ ]:
missing_values = combined.isnull().sum()

print("Missing Values:")
print(missing_values)

print("\nTotal Missing Values:", missing_values.sum())

In [ ]:
# Check duplicate metadata rows

duplicate_rows = combined.duplicated().sum()

print("Number of Duplicate Rows:", duplicate_rows)

# Show duplicate rows (if any)
if duplicate_rows > 0:
    display(combined[combined.duplicated()].head())

In [ ]:
import matplotlib.pyplot as plt

# Count images from each dataset
dataset_counts = combined['dataset'].value_counts()

print(dataset_counts)

# Plot dataset distribution
plt.figure(figsize=(6,5))
dataset_counts.plot(kind='bar')

plt.title("Dataset Distribution")
plt.xlabel("Dataset")
plt.ylabel("Number of Images")

plt.xticks(rotation=0)

plt.show()

In [ ]:
combined["dataset"] = combined["dataset"].replace({
    "isic2024": "ISIC 2024",
    "HAM10000": "HAM10000",
    "dermacon_in": "DermaCon-IN"
})

In [ ]:
import matplotlib.pyplot as plt

# Count images in each cancer class
class_counts = combined["class_name"].value_counts()

print("Cancer Class Distribution:")
print(class_counts)

# Plot
plt.figure(figsize=(8,5))
class_counts.plot(kind="bar")

plt.title("Cancer Class Distribution")
plt.xlabel("Cancer Type")
plt.ylabel("Number of Images")

plt.xticks(rotation=45)
plt.show()

In [ ]:
# Standardize cancer class names

combined["class_name"] = combined["class_name"].replace({
    "mel": "Melanoma",
    "melanoma": "Melanoma",
    "Melanoma": "Melanoma",

    "bcc": "Basal Cell Carcinoma",
    "basal cell carcinoma": "Basal Cell Carcinoma",
    "Basal Cell Carcinoma": "Basal Cell Carcinoma",

    "squamous cell carcinoma": "Squamous Cell Carcinoma",
    "Squamous Cell Carcinoma": "Squamous Cell Carcinoma",

    "mcc": "Merkel Cell Carcinoma",
    "merkel cell carcinoma": "Merkel Cell Carcinoma",
    "Merkel Cell Carcinoma": "Merkel Cell Carcinoma"
})

In [ ]:
class_counts = combined["class_name"].value_counts()

print(class_counts)

In [ ]:
combined["class_name"] = combined["class_name"].replace({
    "melanoma metastasis": "Melanoma"
})

In [ ]:
# Merge metastatic melanoma into Melanoma

combined["class_name"] = combined["class_name"].replace({
    "melanoma metastasis": "Melanoma"
})

# Check final class distribution
class_counts = combined["class_name"].value_counts()

print(class_counts)

In [ ]:
import os

# Check Kaggle working directory
print("Files and folders in /kaggle/working:")

for item in os.listdir("/kaggle/working"):
    print(item)

In [ ]:
mcc_path = "/kaggle/input/datasets/quantumcoders05/mcc-dataset"  

for root, dirs, files in os.walk(mcc_path):
    print("\nFolder:", root)
    print("Number of files:", len(files))
    print("Sample files:", files[:5])

In [ ]:
from PIL import Image
import os

mcc_path = "/kaggle/input/datasets/quantumcoders05/mcc-dataset/Merkel Cell Carcinoma in Skin of Color Patients A Case Series"

mcc_files = [
    os.path.join(mcc_path, f)
    for f in os.listdir(mcc_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Number of MCC image files:", len(mcc_files))

for path in mcc_files:
    try:
        img = Image.open(path)
        print(os.path.basename(path), "→", img.size, "→", img.mode)
    except Exception as e:
        print("Error:", os.path.basename(path), e)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, path in zip(axes.ravel(), mcc_files):
    img = Image.open(path).convert("RGB")
    ax.imshow(img)
    ax.set_title(os.path.basename(path), fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
mcc_df = pd.DataFrame({
    "image_name": [os.path.basename(path) for path in mcc_files],
    "class_name": ["MCC"] * len(mcc_files),
    "dataset": ["MCC"] * len(mcc_files),
    "image_path": mcc_files
})

mcc_df.head()

In [ ]:
import pandas as pd
import os

mcc_df = pd.DataFrame({
    "image_name": [os.path.basename(path) for path in mcc_files],
    "class_name": ["MCC"] * len(mcc_files),
    "age": [None] * len(mcc_files),
    "sex": [None] * len(mcc_files),
    "dataset": ["MCC"] * len(mcc_files),
    "label": [3] * len(mcc_files),
    "image_path": mcc_files
})

print("MCC Dataset Shape:", mcc_df.shape)
display(mcc_df.head())

In [ ]:
combined["image_path"] = None

In [ ]:
print(combined.columns)

In [ ]:
combined = pd.concat(
    [combined, mcc_df],
    ignore_index=True
)

print("Updated Dataset Shape:", combined.shape)

In [ ]:
print("Final Class Distribution:")
print(combined["class_name"].value_counts())

In [ ]:
import os

combined["image_exists"] = combined["image_path"].apply(
    lambda x: os.path.exists(x) if pd.notna(x) else False
)

print(combined["image_exists"].value_counts())

In [ ]:
print(
    combined.groupby("dataset")["image_exists"]
    .value_counts()
)

In [ ]:
from PIL import Image
import os

mcc_path = "/kaggle/input/datasets/quantumcoders05/mcc-dataset/Merkel Cell Carcinoma in Skin of Color Patients A Case Series"

# Only the 2 required images
required_images = [
    "Supplemental Figure 2a.png",
    "Supplemental Figure 1a.jpg"
]

for filename in required_images:
    path = os.path.join(mcc_path, filename)

    try:
        img = Image.open(path)
        print(filename, "→", img.size, "→", img.mode)
    except Exception as e:
        print("Error:", filename, e)

In [ ]:
# Select only the 2 required MCC images

selected_mcc_files = [
    "Supplemental Figure 2a.png",
    "Supplemental Figure 1a.jpg"
]

# Create full paths
selected_mcc_paths = [
    os.path.join(mcc_path, filename)
    for filename in selected_mcc_files
]

# Remove all existing MCC records
combined = combined[combined["class_name"] != "MCC"].copy()

# Create new MCC records
mcc_selected = pd.DataFrame({
    "image_path": selected_mcc_paths,
    "class_name": ["MCC", "MCC"],
    "dataset": ["MCC", "MCC"]
})

# Add selected MCC images
combined = pd.concat([combined, mcc_selected], ignore_index=True)

print("Updated Class Distribution:")
print(combined["class_name"].value_counts())

In [ ]:
print(combined[combined["class_name"] == "MCC"][["image_path", "class_name"]])

In [ ]:
import os

combined["image_exists"] = combined["image_path"].apply(
    lambda x: os.path.exists(x) if pd.notna(x) else False
)

print(combined.groupby("class_name")["image_exists"].value_counts())

In [ ]:
print(combined.columns.tolist())

In [ ]:
print(combined.head())

In [ ]:
import os

base_path = "/kaggle/input"

for root, dirs, files in os.walk(base_path):
    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
    
    if image_files:
        print("Folder:", root)
        print("Number of images:", len(image_files))
        print("Sample:", image_files[:5])
        print("-" * 60)

In [ ]:
# ============================================================
#  Find all actual image files in the 3 main datasets
# ============================================================

import os

def get_image_files(folder):
    image_files = []

    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                image_files.append(os.path.join(root, file))

    return image_files


# Dataset folders
ham_folder = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
isic_folder = "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629"
derma_folder = "/kaggle/input/datasets/hiro002/dermacon-in-dataset"


# Collect image files
ham_images = get_image_files(ham_folder)
isic_images = get_image_files(isic_folder)
derma_images = get_image_files(derma_folder)


print("HAM10000 images :", len(ham_images))
print("ISIC 2024 images:", len(isic_images))
print("DermaCon-IN images:", len(derma_images))

In [ ]:
# Create image-name to actual file-path mappings

ham_image_map = {}

for path in ham_images:
    filename = os.path.basename(path)
    image_id = os.path.splitext(filename)[0]
    ham_image_map[image_id] = path


isic_image_map = {}

for path in isic_images:
    filename = os.path.basename(path)
    image_id = os.path.splitext(filename)[0]
    isic_image_map[image_id] = path


derma_image_map = {}

for path in derma_images:
    filename = os.path.basename(path)
    image_id = os.path.splitext(filename)[0]
    derma_image_map[image_id] = path


print("HAM10000 mapping :", len(ham_image_map))
print("ISIC 2024 mapping:", len(isic_image_map))
print("DermaCon-IN mapping:", len(derma_image_map))

In [ ]:
# Assign actual image paths to the combined dataset

def find_image_path(row):
    image_name = str(row["image_name"])
    dataset = row["dataset"]

    if dataset == "HAM10000":
        return ham_image_map.get(image_name)

    elif dataset == "ISIC 2024":
        return isic_image_map.get(image_name)

    elif dataset == "DermaCon-IN":
        return derma_image_map.get(image_name)

    elif dataset == "MCC":
        return row["image_path"]

    return None


combined["image_path"] = combined.apply(find_image_path, axis=1)

# Check whether paths were successfully assigned
combined["image_exists"] = combined["image_path"].apply(
    lambda x: os.path.isfile(x) if pd.notna(x) else False
)

print("Image path matching results:")
print(combined.groupby("dataset")["image_exists"].value_counts())

print("\nTotal records:", len(combined))
print("Images with valid paths:", combined["image_exists"].sum())
print("Images without valid paths:", (~combined["image_exists"]).sum())

In [ ]:
# Recreate the combined dataframe

ham_new = pd.DataFrame()

ham_new["image_name"] = ham["image_id"]
ham_new["class_name"] = ham["dx"]
ham_new["age"] = ham["age"]
ham_new["sex"] = ham["sex"]
ham_new["dataset"] = "HAM10000"

ham_mapping = {
    "mel": 0,
    "bcc": 1,
    "scc": 2
}

ham_new["label"] = ham_new["class_name"].map(ham_mapping)
ham_new = ham_new.dropna(subset=["label"])


isic_cancer = isic[
    isic["diagnosis"].str.contains(
        "melanoma|basal cell carcinoma|squamous cell carcinoma",
        case=False,
        na=False
    )
].copy()

isic_new = pd.DataFrame()

isic_new["image_name"] = isic["isic_id"]
isic_new["class_name"] = isic["diagnosis"]
isic_new["age"] = isic["age_approx"]
isic_new["sex"] = isic["sex"]
isic_new["dataset"] = "ISIC 2024"

isic_mapping = {
    "melanoma": 0,
    "melanoma metastasis": 0,
    "basal cell carcinoma": 1,
    "squamous cell carcinoma": 2
}

isic_new["label"] = (
    isic_new["class_name"]
    .str.lower()
    .map(isic_mapping)
)

isic_new = isic_new.dropna(subset=["label"])


derma_cancer = derma[
    derma["Disease_label"].str.contains(
        "Melanoma|Basal Cell Carcinoma|Squamous Cell Carcinoma",
        case=False,
        na=False
    )
].copy()

derma_new = pd.DataFrame()

derma_new["image_name"] = derma["Image_name"]
derma_new["class_name"] = derma["Disease_label"]
derma_new["age"] = derma["Age"]
derma_new["sex"] = derma["Sex"]
derma_new["dataset"] = "DermaCon-IN"

derma_mapping = {
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2
}

derma_new["label"] = derma_new["class_name"].map(derma_mapping)
derma_new = derma_new.dropna(subset=["label"])


# Combine the three datasets

combined = pd.concat(
    [ham_new, isic_new, derma_new],
    ignore_index=True
)

# Add image_path column

combined["image_path"] = None


print("Combined dataset recreated successfully.")
print("Shape:", combined.shape)
print("\nDataset distribution:")
print(combined["dataset"].value_counts())

In [ ]:
# Imports
import os
import numpy as np
import pandas as pd

# Dataset paths
ham_path = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"

isic_path = "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/metadata.csv"

derma_path = "/kaggle/input/datasets/hiro002/dermacon-in-dataset/METADATA/Skin_Metadata.csv"

# Load metadata
ham = pd.read_csv(ham_path)
isic = pd.read_csv(isic_path)
derma = pd.read_csv(derma_path)

print("HAM10000 metadata:", ham.shape)
print("ISIC 2024 metadata:", isic.shape)
print("DermaCon-IN metadata:", derma.shape)

In [ ]:
# Prepare HAM10000 data

ham_new = pd.DataFrame()

ham_new["image_name"] = ham["image_id"]
ham_new["class_name"] = ham["dx"]
ham_new["age"] = ham["age"]
ham_new["sex"] = ham["sex"]
ham_new["dataset"] = "HAM10000"

ham_mapping = {
    "mel": 0,
    "bcc": 1,
    "scc": 2
}

ham_new["label"] = ham_new["class_name"].map(ham_mapping)

ham_new = ham_new.dropna(subset=["label"])


# Prepare ISIC 2024 data

isic_cancer = isic[
    isic["diagnosis"].str.contains(
        "melanoma|basal cell carcinoma|squamous cell carcinoma",
        case=False,
        na=False
    )
].copy()

isic_new = pd.DataFrame()

isic_new["image_name"] = isic_cancer["isic_id"]
isic_new["class_name"] = isic_cancer["diagnosis"]
isic_new["age"] = isic_cancer["age_approx"]
isic_new["sex"] = isic_cancer["sex"]
isic_new["dataset"] = "ISIC 2024"

isic_mapping = {
    "melanoma": 0,
    "melanoma metastasis": 0,
    "basal cell carcinoma": 1,
    "squamous cell carcinoma": 2
}

isic_new["label"] = (
    isic_new["class_name"]
    .str.lower()
    .map(isic_mapping)
)

isic_new = isic_new.dropna(subset=["label"])


# Prepare DermaCon-IN data

derma_cancer = derma[
    derma["Disease_label"].str.contains(
        "Melanoma|Basal Cell Carcinoma|Squamous Cell Carcinoma",
        case=False,
        na=False
    )
].copy()

derma_new = pd.DataFrame()

derma_new["image_name"] = derma_cancer["Image_name"]
derma_new["class_name"] = derma_cancer["Disease_label"]
derma_new["age"] = derma_cancer["Age"]
derma_new["sex"] = derma_cancer["Sex"]
derma_new["dataset"] = "DermaCon-IN"

derma_mapping = {
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2
}

derma_new["label"] = derma_new["class_name"].map(derma_mapping)

derma_new = derma_new.dropna(subset=["label"])


print("HAM10000:", ham_new.shape)
print("ISIC 2024:", isic_new.shape)
print("DermaCon-IN:", derma_new.shape)

In [ ]:
# Check the actual disease labels in DermaCon-IN

print("All DermaCon-IN disease labels:")
print(derma["Disease_label"].value_counts(dropna=False).to_string())

In [ ]:
# Combine the three cancer datasets

combined = pd.concat(
    [ham_new, isic_new, derma_new],
    ignore_index=True
)

# Add image_path column
combined["image_path"] = None


# MCC dataset paths

mcc_path = "/kaggle/input/datasets/quantumcoders05/mcc-dataset/Merkel Cell Carcinoma in Skin of Color Patients A Case Series"

selected_mcc_files = [
    "Supplemental Figure 2a.png",
    "Supplemental Figure 1a.jpg"
]

selected_mcc_paths = [
    os.path.join(mcc_path, filename)
    for filename in selected_mcc_files
]


# Add MCC records

mcc_selected = pd.DataFrame({
    "image_name": selected_mcc_files,
    "class_name": ["Merkel Cell Carcinoma", "Merkel Cell Carcinoma"],
    "age": [None, None],
    "sex": [None, None],
    "dataset": ["MCC", "MCC"],
    "label": [3, 3],
    "image_path": selected_mcc_paths
})


# Add MCC to combined dataset

combined = pd.concat(
    [combined, mcc_selected],
    ignore_index=True
)


print("Combined dataset shape:", combined.shape)

print("\nClass distribution:")
print(combined["class_name"].value_counts())

print("\nDataset distribution:")
print(combined["dataset"].value_counts())

In [ ]:
# Standardize all cancer class names

combined["class_name"] = (
    combined["class_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

combined["class_name"] = combined["class_name"].replace({

    # Melanoma
    "mel": "Melanoma",
    "melanoma": "Melanoma",
    "melanoma metastasis": "Melanoma",

    # Basal Cell Carcinoma
    "bcc": "Basal Cell Carcinoma",
    "basal cell carcinoma": "Basal Cell Carcinoma",

    # Squamous Cell Carcinoma
    "scc": "Squamous Cell Carcinoma",
    "squamous cell carcinoma": "Squamous Cell Carcinoma",

    # Merkel Cell Carcinoma
    "mcc": "Merkel Cell Carcinoma",
    "merkel cell carcinoma": "Merkel Cell Carcinoma"
})


# Assign final numeric labels

final_mapping = {
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2,
    "Merkel Cell Carcinoma": 3
}

combined["label"] = combined["class_name"].map(final_mapping)


# Check final classes

print("Final Class Distribution:")
print(combined["class_name"].value_counts())

print("\nFinal Label Distribution:")
print(combined["label"].value_counts().sort_index())

print("\nMissing labels:", combined["label"].isna().sum())

In [ ]:
# Match each image name with its actual image path

def get_image_path(row):
    
    image_name = str(row["image_name"])
    dataset = row["dataset"]

    if dataset == "HAM10000":
        return ham_image_map.get(image_name)

    elif dataset == "ISIC 2024":
        return isic_image_map.get(image_name)

    elif dataset == "DermaCon-IN":
        return derma_image_map.get(image_name)

    elif dataset == "MCC":
        return row["image_path"]

    return None


combined["image_path"] = combined.apply(
    get_image_path,
    axis=1
)


# Check whether each path actually exists

combined["image_exists"] = combined["image_path"].apply(
    lambda x: os.path.isfile(x) if pd.notna(x) else False
)


print("Image path matching:")
print(
    combined.groupby("dataset")["image_exists"]
    .value_counts()
)


print("\nTotal records:", len(combined))

print(
    "Valid image paths:",
    combined["image_exists"].sum()
)

print(
    "Missing image paths:",
    (~combined["image_exists"]).sum()
)

In [ ]:
# Check DermaCon-IN image names and metadata image names

print("Metadata image names:")
print(derma_cancer["Image_name"].head(20).tolist())

print("\nActual DermaCon-IN image filenames:")
print(
    [os.path.basename(path) for path in derma_images[:20]]
)

In [ ]:
# Find all files in the DermaCon-IN dataset
# and look for files that may contain image-name mappings

derma_base = "/kaggle/input/datasets/hiro002/dermacon-in-dataset"

for root, dirs, files in os.walk(derma_base):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
# Check DermaCon-IN train/test split metadata

train_split_path = "/kaggle/input/datasets/hiro002/dermacon-in-dataset/METADATA/train_split.csv"
test_split_path = "/kaggle/input/datasets/hiro002/dermacon-in-dataset/METADATA/test_split.csv"

train_split = pd.read_csv(train_split_path)
test_split = pd.read_csv(test_split_path)

print("TRAIN SPLIT")
print("Shape:", train_split.shape)
print(train_split.columns.tolist())
display(train_split.head())

print("\nTEST SPLIT")
print("Shape:", test_split.shape)
print(test_split.columns.tolist())
display(test_split.head())

In [ ]:
# Check whether some DermaCon-IN metadata images exist in /kaggle/input

import os

target_names = set(derma["Image_name"].dropna().astype(str).head(20))

found = {}

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file in target_names:
            found[file] = os.path.join(root, file)

print("Target image names:", len(target_names))
print("Images found:", len(found))

for name, path in found.items():
    print(name, "→", path)

In [ ]:
import os
import pandas as pd

# DermaCon-IN metadata path
derma_path = "/kaggle/input/datasets/hiro002/dermacon-in-dataset/METADATA/Skin_Metadata.csv"

# Load metadata
derma = pd.read_csv(derma_path)

# Get 20 image names from metadata
target_names = set(
    derma["Image_name"]
    .dropna()
    .astype(str)
    .head(20)
)

print("Target image names:", len(target_names))

# Search for these exact filenames
found = {}

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file in target_names:
            found[file] = os.path.join(root, file)

print("Images found:", len(found))

for name, path in found.items():
    print(name, "→", path)

In [ ]:
# Create a complete image-name → image-path mapping for DermaCon-IN

derma_image_map = {}

for root, dirs, files in os.walk(
    "/kaggle/input/datasets/hiro002/dermacon-in-dataset/DATASET"
):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            derma_image_map[file] = os.path.join(root, file)

print("Total DermaCon-IN images found:", len(derma_image_map))


# Assign the correct image path to DermaCon-IN records

derma_new["image_path"] = derma_new["image_name"].map(
    derma_image_map
)


# Check matching

print("\nDermaCon-IN image matching:")
print(
    derma_new["image_path"].notna().value_counts()
)

print(
    "\nMissing image paths:",
    derma_new["image_path"].isna().sum()
)


# Show the 17 cancer images and their paths

display(
    derma_new[
        ["image_name", "class_name", "image_path"]
    ]
)

In [ ]:
# Recreate DermaCon-IN cancer dataframe
# and assign the correct image paths

derma = pd.read_csv(
    "/kaggle/input/datasets/hiro002/dermacon-in-dataset/METADATA/Skin_Metadata.csv"
)

# Select only the 3 cancer types we need from DermaCon-IN

derma_cancer = derma[
    derma["Disease_label"].str.contains(
        "Melanoma|Basal Cell Carcinoma|Squamous Cell Carcinoma",
        case=False,
        na=False
    )
].copy()


# Create standardized dataframe

derma_new = pd.DataFrame()

derma_new["image_name"] = derma_cancer["Image_name"].values
derma_new["class_name"] = derma_cancer["Disease_label"].values
derma_new["age"] = derma_cancer["Age"].values
derma_new["sex"] = derma_cancer["Sex"].values
derma_new["dataset"] = "DermaCon-IN"


# Standardize class names

derma_new["class_name"] = (
    derma_new["class_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

derma_new["class_name"] = derma_new["class_name"].replace({
    "melanoma": "Melanoma",
    "basal cell carcinoma": "Basal Cell Carcinoma",
    "squamous cell carcinoma": "Squamous Cell Carcinoma"
})


# Assign labels

derma_mapping = {
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2
}

derma_new["label"] = derma_new["class_name"].map(derma_mapping)


# Assign actual image paths

derma_new["image_path"] = derma_new["image_name"].map(
    derma_image_map
)


# Check results

print("DermaCon-IN cancer records:", len(derma_new))

print("\nClass distribution:")
print(derma_new["class_name"].value_counts())

print("\nImage path matching:")
print(derma_new["image_path"].notna().value_counts())

print("\nMissing image paths:")
print(derma_new["image_path"].isna().sum())

print("\nDermaCon-IN records:")
display(
    derma_new[
        ["image_name", "class_name", "label", "image_path"]
    ]
)

In [ ]:
# Rebuild the final combined dataset
# using HAM10000 + ISIC 2024 + DermaCon-IN + MCC

# Make sure MCC image paths are available

mcc_path = "/kaggle/input/datasets/quantumcoders05/mcc-dataset/Merkel Cell Carcinoma in Skin of Color Patients A Case Series"

selected_mcc_files = [
    "Supplemental Figure 2a.png",
    "Supplemental Figure 1a.jpg"
]

selected_mcc_paths = [
    os.path.join(mcc_path, filename)
    for filename in selected_mcc_files
]


# Create MCC dataframe

mcc_selected = pd.DataFrame({
    "image_name": selected_mcc_files,
    "class_name": ["Merkel Cell Carcinoma", "Merkel Cell Carcinoma"],
    "age": [None, None],
    "sex": [None, None],
    "dataset": ["MCC", "MCC"],
    "label": [3, 3],
    "image_path": selected_mcc_paths
})


# Make sure HAM and ISIC have image_path columns

ham_new["image_path"] = None
isic_new["image_path"] = None


# Create image maps for HAM10000 and ISIC 2024

ham_image_map = {}

for root, dirs, files in os.walk(
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            ham_image_map[file] = os.path.join(root, file)


isic_image_map = {}

for root, dirs, files in os.walk(
    "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629"
):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            isic_image_map[file] = os.path.join(root, file)


# Assign image paths

ham_new["image_path"] = ham_new["image_name"].map(ham_image_map)

isic_new["image_path"] = isic_new["image_name"].map(isic_image_map)


# Combine everything

combined = pd.concat(
    [
        ham_new,
        isic_new,
        derma_new,
        mcc_selected
    ],
    ignore_index=True
)


# Check actual image existence

combined["image_exists"] = combined["image_path"].apply(
    lambda x: os.path.isfile(x) if pd.notna(x) else False
)


print("FINAL COMBINED DATASET")
print("----------------------")

print("Total records:", len(combined))

print("\nClass distribution:")
print(combined["class_name"].value_counts())

print("\nDataset distribution:")
print(combined["dataset"].value_counts())

print("\nImage path validation:")
print(combined["image_exists"].value_counts())

print(
    "\nMissing image paths:",
    (~combined["image_exists"]).sum()
)

In [ ]:
import os
import pandas as pd
import numpy as np


# =========================================================
# 1. LOAD METADATA
# =========================================================

ham_path = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"

isic_path = "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/metadata.csv"

derma_path = "/kaggle/input/datasets/hiro002/dermacon-in-dataset/METADATA/Skin_Metadata.csv"

ham = pd.read_csv(ham_path)

isic = pd.read_csv(
    isic_path,
    low_memory=False
)

derma = pd.read_csv(derma_path)


# =========================================================
# 2. CREATE HAM10000 DATAFRAME
# =========================================================

ham_new = pd.DataFrame()

ham_new["image_name"] = ham["image_id"]
ham_new["class_name"] = ham["dx"]
ham_new["age"] = ham["age"]
ham_new["sex"] = ham["sex"]
ham_new["dataset"] = "HAM10000"

ham_mapping = {
    "mel": 0,
    "bcc": 1,
    "scc": 2
}

ham_new["label"] = ham_new["class_name"].map(ham_mapping)

ham_new = ham_new.dropna(subset=["label"])


# =========================================================
# 3. CREATE ISIC 2024 DATAFRAME
# =========================================================

isic_cancer = isic[
    isic["diagnosis"].str.contains(
        "melanoma|basal cell carcinoma|squamous cell carcinoma",
        case=False,
        na=False
    )
].copy()

isic_new = pd.DataFrame()

isic_new["image_name"] = isic_cancer["isic_id"]
isic_new["class_name"] = isic_cancer["diagnosis"]
isic_new["age"] = isic_cancer["age_approx"]
isic_new["sex"] = isic_cancer["sex"]
isic_new["dataset"] = "ISIC 2024"

isic_mapping = {
    "melanoma": 0,
    "melanoma metastasis": 0,
    "basal cell carcinoma": 1,
    "squamous cell carcinoma": 2
}

isic_new["label"] = (
    isic_new["class_name"]
    .str.lower()
    .map(isic_mapping)
)

isic_new = isic_new.dropna(subset=["label"])


# =========================================================
# 4. CREATE DERMACON-IN DATAFRAME
# =========================================================

derma_cancer = derma[
    derma["Disease_label"].str.contains(
        "Melanoma|Basal Cell Carcinoma|Squamous Cell Carcinoma",
        case=False,
        na=False
    )
].copy()

derma_new = pd.DataFrame()

derma_new["image_name"] = derma_cancer["Image_name"].values
derma_new["class_name"] = derma_cancer["Disease_label"].values
derma_new["age"] = derma_cancer["Age"].values
derma_new["sex"] = derma_cancer["Sex"].values
derma_new["dataset"] = "DermaCon-IN"


# Standardize class names

derma_new["class_name"] = (
    derma_new["class_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

derma_new["class_name"] = derma_new["class_name"].replace({
    "melanoma": "Melanoma",
    "basal cell carcinoma": "Basal Cell Carcinoma",
    "squamous cell carcinoma": "Squamous Cell Carcinoma"
})


derma_mapping = {
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2
}

derma_new["label"] = derma_new["class_name"].map(
    derma_mapping
)


# =========================================================
# 5. CREATE DERMACON-IN IMAGE PATH MAP
# =========================================================

derma_image_map = {}

derma_image_root = (
    "/kaggle/input/datasets/hiro002/dermacon-in-dataset/DATASET"
)

for root, dirs, files in os.walk(derma_image_root):

    for file in files:

        if file.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):

            derma_image_map[file] = os.path.join(
                root,
                file
            )


derma_new["image_path"] = derma_new["image_name"].map(
    derma_image_map
)


# =========================================================
# 6. CREATE HAM10000 IMAGE PATH MAP
# =========================================================

ham_image_map = {}

ham_root = (
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
)

for root, dirs, files in os.walk(ham_root):

    for file in files:

        if file.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):

            ham_image_map[file] = os.path.join(
                root,
                file
            )


ham_new["image_path"] = ham_new["image_name"].map(
    ham_image_map
)


# =========================================================
# 7. CREATE ISIC 2024 IMAGE PATH MAP
# =========================================================

isic_image_map = {}

isic_root = (
    "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629"
)

for root, dirs, files in os.walk(isic_root):

    for file in files:

        if file.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):

            isic_image_map[file] = os.path.join(
                root,
                file
            )


isic_new["image_path"] = isic_new["image_name"].map(
    isic_image_map
)


# =========================================================
# 8. CREATE MCC DATAFRAME
# =========================================================

mcc_path = (
    "/kaggle/input/datasets/quantumcoders05/"
    "mcc-dataset/"
    "Merkel Cell Carcinoma in Skin of Color Patients A Case Series"
)

selected_mcc_files = [
    "Supplemental Figure 2a.png",
    "Supplemental Figure 1a.jpg"
]

selected_mcc_paths = [
    os.path.join(
        mcc_path,
        filename
    )
    for filename in selected_mcc_files
]


mcc_selected = pd.DataFrame({

    "image_name": selected_mcc_files,

    "class_name": [
        "Merkel Cell Carcinoma",
        "Merkel Cell Carcinoma"
    ],

    "age": [
        None,
        None
    ],

    "sex": [
        None,
        None
    ],

    "dataset": [
        "MCC",
        "MCC"
    ],

    "label": [
        3,
        3
    ],

    "image_path": selected_mcc_paths

})


# =========================================================
# 9. COMBINE ALL DATASETS
# =========================================================

combined = pd.concat(
    [
        ham_new,
        isic_new,
        derma_new,
        mcc_selected
    ],
    ignore_index=True
)


# =========================================================
# 10. CHECK IMAGE PATHS
# =========================================================

combined["image_exists"] = combined[
    "image_path"
].apply(
    lambda x: os.path.isfile(x)
    if pd.notna(x)
    else False
)


# =========================================================
# 11. STANDARDIZE FINAL CLASS NAMES
# =========================================================

combined["class_name"] = (
    combined["class_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

combined["class_name"] = combined["class_name"].replace({

    "mel": "Melanoma",
    "melanoma": "Melanoma",
    "melanoma metastasis": "Melanoma",

    "bcc": "Basal Cell Carcinoma",
    "basal cell carcinoma": "Basal Cell Carcinoma",

    "scc": "Squamous Cell Carcinoma",
    "squamous cell carcinoma": "Squamous Cell Carcinoma",

    "mcc": "Merkel Cell Carcinoma",
    "merkel cell carcinoma": "Merkel Cell Carcinoma"
})


# Final labels

final_mapping = {
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2,
    "Merkel Cell Carcinoma": 3
}

combined["label"] = combined[
    "class_name"
].map(final_mapping)


# =========================================================
# 12. FINAL CHECK
# =========================================================

print("========================================")
print("FINAL COMBINED DATASET")
print("========================================")

print("\nTotal records:")
print(len(combined))

print("\nClass distribution:")
print(
    combined["class_name"].value_counts()
)

print("\nDataset distribution:")
print(
    combined["dataset"].value_counts()
)

print("\nImage path validation:")
print(
    combined["image_exists"].value_counts()
)

print("\nMissing image paths:")
print(
    (~combined["image_exists"]).sum()
)

print("\nMissing labels:")
print(
    combined["label"].isna().sum()
)

In [ ]:
import os

# =========================================================
# FIND ACTUAL HAM10000 IMAGE FILES
# =========================================================

ham_root = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"

ham_image_map = {}

for root, dirs, files in os.walk(ham_root):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            # Store filename without extension
            file_id = os.path.splitext(file)[0]
            ham_image_map[file_id] = os.path.join(root, file)

print("HAM10000 actual images found:", len(ham_image_map))


# =========================================================
# FIND ACTUAL ISIC 2024 IMAGE FILES
# =========================================================

isic_root = "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629"

isic_image_map = {}

for root, dirs, files in os.walk(isic_root):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            file_id = os.path.splitext(file)[0]
            isic_image_map[file_id] = os.path.join(root, file)

print("ISIC 2024 actual images found:", len(isic_image_map))


# =========================================================
# ASSIGN HAM PATHS
# =========================================================

ham_mask = combined["dataset"] == "HAM10000"

combined.loc[ham_mask, "image_path"] = (
    combined.loc[ham_mask, "image_name"]
    .astype(str)
    .map(ham_image_map)
)


# =========================================================
# ASSIGN ISIC PATHS
# =========================================================

isic_mask = combined["dataset"] == "ISIC 2024"

combined.loc[isic_mask, "image_path"] = (
    combined.loc[isic_mask, "image_name"]
    .astype(str)
    .map(isic_image_map)
)


# =========================================================
# CHECK IMAGE EXISTENCE AGAIN
# =========================================================

combined["image_exists"] = combined["image_path"].apply(
    lambda x: os.path.isfile(x)
    if pd.notna(x)
    else False
)


# =========================================================
# RESULTS
# =========================================================

print("\n=======================================")
print("IMAGE PATH MATCHING RESULT")
print("=======================================")

print("\nBy Dataset:")
print(
    combined.groupby("dataset")["image_exists"]
    .value_counts()
)

print("\nTotal records:", len(combined))

print(
    "Valid image paths:",
    combined["image_exists"].sum()
)

print(
    "Missing image paths:",
    (~combined["image_exists"]).sum()
)

In [ ]:
# Image validation

from PIL import Image
import os
import pandas as pd

valid_images = []
invalid_images = []

for idx, row in combined.iterrows():

    image_path = row["image_path"]

    try:
        img = Image.open(image_path)
        img.verify()

        valid_images.append(idx)

    except Exception as e:

        invalid_images.append({
            "index": idx,
            "image_path": image_path,
            "error": str(e)
        })


print("=======================================")
print("IMAGE VALIDATION RESULT")
print("=======================================")

print("Total records:", len(combined))
print("Valid images:", len(valid_images))
print("Invalid images:", len(invalid_images))


if len(invalid_images) > 0:

    invalid_df = pd.DataFrame(invalid_images)

    print("\nInvalid images:")
    display(invalid_df.head(20))

else:

    print("\nNo corrupted or unreadable images found.")

In [ ]:
from PIL import Image
import numpy as np

# EfficientNet-B4 input size
IMG_SIZE = 380

def load_and_resize_image(image_path):
    """
    Load image, convert to RGB,
    and resize to 380 x 380.
    """

    image = Image.open(image_path).convert("RGB")

    image = image.resize(
        (IMG_SIZE, IMG_SIZE),
        Image.Resampling.LANCZOS
    )

    return np.array(image)


# Test with a few images

sample_paths = combined["image_path"].head(5)

for path in sample_paths:

    image = load_and_resize_image(path)

    print(
        "Image:",
        os.path.basename(path),
        "| Shape:",
        image.shape,
        "| Data type:",
        image.dtype,
        "| Min:",
        image.min(),
        "| Max:",
        image.max()
    )

In [ ]:
import torch
from torchvision import transforms
from PIL import Image

# EfficientNet-B4 ImageNet normalization

image_transform = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Test normalization on one image

sample_path = combined["image_path"].iloc[0]

image = Image.open(sample_path).convert("RGB")

transformed_image = image_transform(image)

print("Original image size:", image.size)
print("Transformed shape:", transformed_image.shape)
print("Data type:", transformed_image.dtype)
print("Minimum value:", transformed_image.min().item())
print("Maximum value:", transformed_image.max().item())

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image


class SkinCancerDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["image_path"]
        label = int(row["label"])

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Apply preprocessing
        if self.transform is not None:
            image = self.transform(image)

        return image, label


# Create dataset

skin_dataset = SkinCancerDataset(
    dataframe=combined,
    transform=image_transform
)


# Test one sample

image, label = skin_dataset[0]

print("Image tensor shape:", image.shape)
print("Image data type:", image.dtype)
print("Label:", label)
print("Total dataset samples:", len(skin_dataset))

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd


# Separate MCC because it has only 2 images
mcc_data = combined[
    combined["class_name"] == "Merkel Cell Carcinoma"
].copy()

main_data = combined[
    combined["class_name"] != "Merkel Cell Carcinoma"
].copy()


# =========================================================
# TRAIN / TEMP SPLIT
# 80% Train
# 20% Temporary
# =========================================================

train_df, temp_df = train_test_split(
    main_data,
    test_size=0.20,
    random_state=42,
    stratify=main_data["label"]
)


# =========================================================
# VALIDATION / TEST SPLIT
# 10% Validation
# 10% Test
# =========================================================

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)


# =========================================================
# ADD MCC IMAGES TO TRAINING DATA
# =========================================================

train_df = pd.concat(
    [train_df, mcc_data],
    ignore_index=True
)


# Shuffle training data

train_df = train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# =========================================================
# CHECK SPLIT SIZES
# =========================================================

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

print("\nTraining class distribution:")
print(train_df["class_name"].value_counts())

print("\nValidation class distribution:")
print(val_df["class_name"].value_counts())

print("\nTesting class distribution:")
print(test_df["class_name"].value_counts())

In [ ]:
import torch
import numpy as np

# Get training labels
train_labels = train_df["label"].astype(int).values

# Count samples in each class
class_counts = np.bincount(
    train_labels,
    minlength=4
)

print("Class counts:")
for i, count in enumerate(class_counts):
    print(f"Class {i}: {count}")


# Calculate class weights
total_samples = len(train_labels)
num_classes = 4

class_weights = total_samples / (
    num_classes * class_counts
)

print("\nClass weights:")
for i, weight in enumerate(class_weights):
    print(f"Class {i}: {weight:.4f}")


# Convert to PyTorch tensor
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print("\nPyTorch class weights:")
print(class_weights)

In [ ]:
# Keep the calculated class counts for reference

print("Training class counts:")
print(class_counts)

print("\nCalculated class weights:")
print(class_weights)

In [ ]:
from torchvision import transforms


# =========================================================
# TRAINING TRANSFORM
# =========================================================

train_transform = transforms.Compose([

    # Resize image for EfficientNet-B4
    transforms.Resize((380, 380)),

    # Random horizontal flip
    transforms.RandomHorizontalFlip(p=0.5),

    # Small random rotation
    transforms.RandomRotation(degrees=15),

    # Random crop and resize
    transforms.RandomResizedCrop(
        size=380,
        scale=(0.85, 1.0)
    ),

    # Convert image to Tensor
    transforms.ToTensor(),

    # ImageNet normalization
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# =========================================================
# VALIDATION TRANSFORM
# =========================================================

val_transform = transforms.Compose([

    transforms.Resize((380, 380)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# =========================================================
# TEST TRANSFORM
# =========================================================

test_transform = transforms.Compose([

    transforms.Resize((380, 380)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


print("Training transform:")
print(train_transform)

print("\nValidation transform:")
print(val_transform)

print("\nTest transform:")
print(test_transform)

In [ ]:
from torch.utils.data import DataLoader


# Create datasets

train_dataset = SkinCancerDataset(
    dataframe=train_df,
    transform=train_transform
)

val_dataset = SkinCancerDataset(
    dataframe=val_df,
    transform=val_transform
)

test_dataset = SkinCancerDataset(
    dataframe=test_df,
    transform=test_transform
)


# Create DataLoaders

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# Check one batch

images, labels = next(iter(train_loader))

print("Train batch image shape:", images.shape)
print("Train batch labels shape:", labels.shape)

print("Batch size:", images.size(0))
print("Image channels:", images.size(1))
print("Image height:", images.size(2))
print("Image width:", images.size(3))

print("\nLabels in first batch:")
print(labels)

print("\nNumber of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of testing batches:", len(test_loader))

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights


# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# Load pretrained EfficientNet-B4
weights = EfficientNet_B4_Weights.DEFAULT

model = efficientnet_b4(weights=weights)


# Number of input features of the original classifier
num_features = model.classifier[1].in_features

print("Original classifier input features:", num_features)


# Replace the original classifier
# Our project has 4 classes:
# 0 - Melanoma
# 1 - Basal Cell Carcinoma
# 2 - Squamous Cell Carcinoma
# 3 - Merkel Cell Carcinoma

model.classifier[1] = nn.Linear(
    num_features,
    4
)


# Move model to GPU/CPU
model = model.to(device)


print("\nEfficientNet-B4:")
print(model.classifier)

print("\nNumber of classes:", 4)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


# Loss function
criterion = nn.CrossEntropyLoss()


# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)


print("Loss function:")
print(criterion)

print("\nOptimizer:")
print(optimizer)

print("\nInitial learning rate:")
print(optimizer.param_groups[0]["lr"])

print("\nScheduler:")
print(scheduler)

In [ ]:
import torch
import os

# Number of epochs
NUM_EPOCHS = 10

# Best validation loss
best_val_loss = float("inf")

# Store training history
history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

# Folder to save model
os.makedirs("/kaggle/working/models", exist_ok=True)

best_model_path = "/kaggle/working/models/best_efficientnet_b4.pth"


for epoch in range(NUM_EPOCHS):

    # =====================================================
    # TRAINING
    # =====================================================

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()


    train_loss = running_loss / total
    train_accuracy = 100 * correct / total


    # =====================================================
    # VALIDATION
    # =====================================================

    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)

            # Validation loss
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()


    val_loss = val_running_loss / val_total
    val_accuracy = 100 * val_correct / val_total


    # =====================================================
    # UPDATE LEARNING RATE
    # =====================================================

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]


    # =====================================================
    # SAVE HISTORY
    # =====================================================

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)

    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)


    # =====================================================
    # SAVE BEST MODEL
    # =====================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            best_model_path
        )

        best_status = "✓ Best model saved"

    else:

        best_status = ""


    # =====================================================
    # PRINT RESULTS
    # =====================================================

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2f}% | "
        f"LR: {current_lr:.6f} "
        f"{best_status}"
    )


print("\n=======================================")
print("TRAINING COMPLETED")
print("=======================================")

print("Best Validation Loss:", best_val_loss)
print("Best model saved at:")
print(best_model_path)

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))

In [ ]:
!nvidia-smi

In [ ]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))

In [ ]:
!pip install -q --force-reinstall \
    torch==2.5.1 \
    torchvision==0.20.1 \
    --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

In [ ]:
!pip uninstall -y torch torchvision torchaudio

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 \
    --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 \
    --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

In [ ]:
!pip install --force-reinstall --no-cache-dir \
    torch==2.5.1 torchvision==0.20.1 \
    --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))                                                               

In [ ]:
import sys
!{sys.executable} -m pip show torch torchvision

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = EfficientNet_B4_Weights.DEFAULT
model = efficientnet_b4(weights=weights)

# Replace classifier for 4 classes
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 4)

model = model.to(device)

print("Device:", device)
print("Model:", model.classifier)

In [ ]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Labels shape:", labels.shape)

In [ ]:
print(len(train_loader))
print(len(val_loader))
print(len(test_loader))

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Custom Dataset
class SkinCancerDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label


# Create datasets
train_dataset = SkinCancerDataset(
    train_df,
    transform=train_transform
)

val_dataset = SkinCancerDataset(
    val_df,
    transform=val_transform
)

test_dataset = SkinCancerDataset(
    test_df,
    transform=test_transform
)


# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of testing batches:", len(test_loader))

In [ ]:
from sklearn.model_selection import train_test_split

# Make sure the final combined dataframe is available
print("Total records:", len(combined_df))

# Stratified split: 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(
    combined_df,
    test_size=0.20,
    stratify=combined_df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

print("\nTraining class distribution:")
print(train_df["class_name"].value_counts())

print("\nValidation class distribution:")
print(val_df["class_name"].value_counts())

print("\nTesting class distribution:")
print(test_df["class_name"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

print("Total records:", len(combined_df))

# 80% Train, 10% Validation, 10% Test
train_df, temp_df = train_test_split(
    combined_df,
    test_size=0.20,
    stratify=combined_df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

print("\nTraining class distribution:")
print(train_df["class_name"].value_counts())

print("\nValidation class distribution:")
print(val_df["class_name"].value_counts())

print("\nTesting class distribution:")
print(test_df["class_name"].value_counts())

In [ ]:
combined_df = pd.concat(
    [ham_df, isic_df, dermacon_df, mcc_df],
    ignore_index=True
)

In [ ]:
import pandas as pd

combined_df = pd.concat(
    [ham_df, isic_df, dermacon_df, mcc_df],
    ignore_index=True
)

print("=======================================")
print("FINAL COMBINED DATASET")
print("=======================================")

print("Total records:", len(combined_df))

print("\nClass distribution:")
print(combined_df["class_name"].value_counts())

print("\nDataset distribution:")
print(combined_df["dataset"].value_counts())

In [ ]:
ham_path="/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"

ham=pd.read_csv(ham_path)

print(ham.head())

In [ ]:
ham_df = ham.copy()

# Rename columns to match the combined dataset structure
ham_df["image_name"] = ham_df["image_id"] + ".jpg"

# HAM10000 class mapping
ham_class_map = {
    "mel": "Melanoma",
    "bcc": "Basal Cell Carcinoma",
    "scc": "Squamous Cell Carcinoma"
}

ham_df["class_name"] = ham_df["dx"].map(ham_class_map)

# Keep only the 3 required cancer classes
ham_df = ham_df[ham_df["class_name"].notna()].copy()

# Numeric labels
class_to_label = {
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2,
    "Merkel Cell Carcinoma": 3
}

ham_df["label"] = ham_df["class_name"].map(class_to_label)

# Dataset name
ham_df["dataset"] = "HAM10000"

print("HAM10000 cancer records:", len(ham_df))
print("\nClass distribution:")
print(ham_df["class_name"].value_counts())

print("\nColumns:")
print(ham_df.columns.tolist())

In [ ]:
import os

# HAM10000 image directories
ham_dirs = [
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2"
]

# Create image name → full path mapping
ham_image_map = {}

for folder in ham_dirs:
    if os.path.exists(folder):
        for file in os.listdir(folder):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                ham_image_map[file] = os.path.join(folder, file)

print("HAM10000 actual images found:", len(ham_image_map))

# Assign image paths
ham_df["image_path"] = ham_df["image_name"].map(ham_image_map)

# Check matching
print("\n=======================================")
print("HAM10000 IMAGE PATH MATCHING")
print("=======================================")

print("Total HAM10000 cancer records:", len(ham_df))
print("Valid image paths:", ham_df["image_path"].notna().sum())
print("Missing image paths:", ham_df["image_path"].isna().sum())

In [ ]:
# Combine all datasets

combined_df = pd.concat(
    [ham_df, isic_df, dermacon_df, mcc_df],
    ignore_index=True
)

print("=======================================")
print("FINAL COMBINED DATASET")
print("=======================================")

print("Total records:", len(combined_df))

print("\nClass distribution:")
print(combined_df["class_name"].value_counts())

print("\nDataset distribution:")
print(combined_df["dataset"].value_counts())

print("\nColumns:")
print(combined_df.columns.tolist())

In [ ]:
import os
import pandas as pd

# Find ISIC 2024 metadata CSV
isic_base = "/kaggle/input/datasets"

for root, dirs, files in os.walk(isic_base):
    for file in files:
        if file.lower().endswith(".csv") and "isic" in file.lower():
            print(os.path.join(root, file))

In [ ]:
import os

base_path = "/kaggle/input/datasets"

csv_files = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("CSV files found:", len(csv_files))

for path in csv_files:
    print(path)

In [9]:
isic_path = "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/metadata.csv"

isic = pd.read_csv(isic_path)

print(isic.head())
print("\nColumns:")
print(isic.columns.tolist())

print("\nShape:")
print(isic.shape)

        isic_id                             attribution copyright_license  \
0  ISIC_7559201  Memorial Sloan Kettering Cancer Center             CC-BY   
1  ISIC_0485014  Memorial Sloan Kettering Cancer Center             CC-BY   
2  ISIC_5257439  Memorial Sloan Kettering Cancer Center             CC-BY   
3  ISIC_2989732  Memorial Sloan Kettering Cancer Center             CC-BY   
4  ISIC_5638210  Memorial Sloan Kettering Cancer Center             CC-BY   

   acquisition_day  age_approx anatom_site_general benign_malignant  \
0           2497.0        55.0      anterior torso           benign   
1              1.0        45.0     lower extremity           benign   
2           2360.0        40.0       lateral torso           benign   
3             78.0        80.0      anterior torso           benign   
4             78.0        80.0      anterior torso           benign   

   clin_size_long_diam_mm concomitant_biopsy       dermoscopic_type  ...  \
0                     6.6         

/tmp/ipykernel_58/1363520295.py:3: DtypeWarning: Columns (8,13,16,17,19) have mixed types. Specify dtype option on import or set low_memory=False.
  isic = pd.read_csv(isic_path)


In [ ]:
# Prepare ISIC 2024 dataframe

print("ISIC total records:", len(isic))

print("\nBenign / Malignant distribution:")
print(isic["benign_malignant"].value_counts(dropna=False))

print("\nDiagnosis distribution:")
print(isic["diagnosis"].value_counts(dropna=False).head(20))

In [ ]:
import os
import pandas as pd

# ISIC metadata
isic_path = "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/metadata.csv"

isic = pd.read_csv(isic_path, low_memory=False)

# Keep only Melanoma and Basal Cell Carcinoma
isic_df = isic[
    isic["diagnosis"].isin([
        "melanoma",
        "basal cell carcinoma"
    ])
].copy()

# Create class names
isic_df["class_name"] = isic_df["diagnosis"].map({
    "melanoma": "Melanoma",
    "basal cell carcinoma": "Basal Cell Carcinoma"
})

# Labels
isic_df["label"] = isic_df["class_name"].map({
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1
})

# Dataset name
isic_df["dataset"] = "ISIC"

print("=======================================")
print("ISIC CANCER DATASET")
print("=======================================")

print("Total cancer records:", len(isic_df))

print("\nClass distribution:")
print(isic_df["class_name"].value_counts())

print("\nColumns:")
print(isic_df.columns.tolist())

In [ ]:
print("dermacon_df exists:", "dermacon_df" in globals())
print("mcc_df exists:", "mcc_df" in globals())

In [ ]:
import pandas as pd

# DermaCon-IN metadata
derma_path = "/kaggle/input/datasets/hiro002/dermacon-in-dataset/METADATA/Skin_Metadata.csv"

derma = pd.read_csv(derma_path)

print("DermaCon-IN total records:", len(derma))
print("\nColumns:")
print(derma.columns.tolist())

print("\nFirst 5 records:")
print(derma.head())

In [ ]:
# Create DermaCon-IN cancer dataframe

dermacon_df = derma[
    derma["Disease_label"].isin([
        "Basal Cell Carcinoma",
        "Squamous Cell Carcinoma",
        "Melanoma"
    ])
].copy()

# Class labels
class_to_label = {
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2,
    "Merkel Cell Carcinoma": 3
}

dermacon_df["class_name"] = dermacon_df["Disease_label"]
dermacon_df["label"] = dermacon_df["class_name"].map(class_to_label)

# Rename image column
dermacon_df["image_name"] = dermacon_df["Image_name"]

# Dataset name
dermacon_df["dataset"] = "DermaCon-IN"

print("=======================================")
print("DERMACON-IN CANCER DATASET")
print("=======================================")

print("Cancer records:", len(dermacon_df))

print("\nClass distribution:")
print(dermacon_df["class_name"].value_counts())

print("\nColumns:")
print(dermacon_df.columns.tolist())

In [ ]:
import os

# DermaCon-IN image directory
derma_image_dir = "/kaggle/input/datasets/hiro002/dermacon-in-dataset/DATASET"

# Create image name → full path mapping
derma_image_map = {}

for root, dirs, files in os.walk(derma_image_dir):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            derma_image_map[file] = os.path.join(root, file)

print("Total DermaCon-IN images found:", len(derma_image_map))

# Match image paths
dermacon_df["image_path"] = dermacon_df["image_name"].map(derma_image_map)

print("\n=======================================")
print("DERMACON-IN IMAGE PATH MATCHING")
print("=======================================")

print("Total records:", len(dermacon_df))
print("Valid image paths:", dermacon_df["image_path"].notna().sum())
print("Missing image paths:", dermacon_df["image_path"].isna().sum())

if dermacon_df["image_path"].isna().sum() > 0:
    print("\nMissing images:")
    print(dermacon_df.loc[
        dermacon_df["image_path"].isna(),
        "image_name"
    ].tolist())

In [ ]:
import os
import pandas as pd

# Find MCC files
mcc_files = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith((".csv", ".xlsx", ".xls")):
            if "mcc" in file.lower() or "merkel" in file.lower():
                mcc_files.append(os.path.join(root, file))

print("MCC files found:")

for f in mcc_files:
    print(f)

In [ ]:
import os

print("=======================================")
print("KAGGLE INPUT FILES")
print("=======================================")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    
    if level <= 2:
        print("\nFolder:", root)
        
        for file in files:
            print("  ", file)

In [ ]:
import os

print("=======================================")
print("ALL DATASET FILES")
print("=======================================")

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import os

print("=======================================")
print("DATASET FOLDERS")
print("=======================================")

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    # folders only
    print(root)

In [ ]:
import os

mcc_base = "/kaggle/input/datasets/quantumcoders05/mcc-dataset"

print("=======================================")
print("MCC DATASET FILES")
print("=======================================")

for root, dirs, files in os.walk(mcc_base):
    print("\nFolder:", root)

    # Only print non-image files
    for file in files:
        if not file.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            print("  FILE:", file)

In [ ]:
import os
import pandas as pd

# MCC image folder
mcc_path = "/kaggle/input/datasets/quantumcoders05/mcc-dataset/Merkel Cell Carcinoma in Skin of Color Patients A Case Series"

# Find image files
mcc_images = []

for root, dirs, files in os.walk(mcc_path):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            mcc_images.append(os.path.join(root, file))

print("=======================================")
print("MCC DATASET")
print("=======================================")

print("MCC images found:", len(mcc_images))

print("\nFirst 10 images:")
for path in mcc_images[:10]:
    print(os.path.basename(path))

In [ ]:
import pandas as pd
import os

# Create MCC dataframe from the 6 images
mcc_df = pd.DataFrame({
    "image_path": mcc_images,
    "image_name": [os.path.basename(x) for x in mcc_images],
    "class_name": ["Merkel Cell Carcinoma"] * len(mcc_images),
    "label": [3] * len(mcc_images),
    "dataset": ["MCC"] * len(mcc_images)
})

print("=======================================")
print("MCC DATAFRAME")
print("=======================================")

print("MCC records:", len(mcc_df))
print("\nClass distribution:")
print(mcc_df["class_name"].value_counts())

print("\nMCC dataframe:")
print(mcc_df[["image_name", "class_name", "label", "dataset"]])

In [ ]:
combined_df = pd.concat(
    [ham_df, isic_df, dermacon_df, mcc_df],
    ignore_index=True
)

print("=======================================")
print("FINAL COMBINED DATASET")
print("=======================================")

print("Total records:", len(combined_df))

print("\nClass distribution:")
print(combined_df["class_name"].value_counts())

print("\nDataset distribution:")
print(combined_df["dataset"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

# 80% Train, 10% Validation, 10% Test
train_df, temp_df = train_test_split(
    combined_df,
    test_size=0.20,
    stratify=combined_df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("=======================================")
print("DATASET SPLIT")
print("=======================================")

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

print("\nTraining class distribution:")
print(train_df["class_name"].value_counts())

print("\nValidation class distribution:")
print(val_df["class_name"].value_counts())

print("\nTesting class distribution:")
print(test_df["class_name"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 80% training, 20% temporary
train_df, temp_df = train_test_split(
    combined_df,
    test_size=0.20,
    stratify=combined_df["label"],
    random_state=42
)

# For validation/test, use stratification only for classes
# with enough samples
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("=======================================")
print("DATASET SPLIT")
print("=======================================")

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

print("\nTraining class distribution:")
print(train_df["class_name"].value_counts())

print("\nValidation class distribution:")
print(val_df["class_name"].value_counts())

print("\nTesting class distribution:")
print(test_df["class_name"].value_counts())

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class SkinCancerDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["image_path"]
        label = int(row["label"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


# Create datasets
train_dataset = SkinCancerDataset(
    train_df,
    transform=train_transform
)

val_dataset = SkinCancerDataset(
    val_df,
    transform=val_transform
)

test_dataset = SkinCancerDataset(
    test_df,
    transform=test_transform
)


print("=======================================")
print("DATASETS CREATED")
print("=======================================")

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

In [ ]:
from torchvision import transforms

# =======================================
# IMAGE TRANSFORMS
# =======================================

train_transform = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(
        size=(380, 380),
        scale=(0.85, 1.0),
        ratio=(0.75, 1.3333)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("=======================================")
print("TRANSFORMS CREATED")
print("=======================================")

print("Training transform:")
print(train_transform)

print("\nValidation transform:")
print(val_transform)

print("\nTest transform:")
print(test_transform)

In [ ]:
# =======================================
# CREATE DATASETS
# =======================================

train_dataset = SkinCancerDataset(
    train_df,
    transform=train_transform
)

val_dataset = SkinCancerDataset(
    val_df,
    transform=val_transform
)

test_dataset = SkinCancerDataset(
    test_df,
    transform=test_transform
)

print("=======================================")
print("DATASETS CREATED")
print("=======================================")

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

In [ ]:
# =======================================
# CREATE DATALOADERS
# =======================================

from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("=======================================")
print("DATALOADERS CREATED")
print("=======================================")

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))
print("Batch size:", BATCH_SIZE)

In [ ]:
# =======================================
# TEST ONE TRAINING BATCH
# =======================================

images, labels = next(iter(train_loader))

print("=======================================")
print("TRAIN BATCH TEST")
print("=======================================")

print("Image batch shape:", images.shape)
print("Labels shape:", labels.shape)

print("Batch size:", images.shape[0])
print("Image channels:", images.shape[1])
print("Image height:", images.shape[2])
print("Image width:", images.shape[3])

print("\nLabels in first batch:")
print(labels)

print("\nUnique labels in batch:")
print(torch.unique(labels))

In [ ]:
# =======================================
# CHECK IMAGE PATHS
# =======================================

print("Train image_path missing:", train_df["image_path"].isna().sum())
print("Validation image_path missing:", val_df["image_path"].isna().sum())
print("Test image_path missing:", test_df["image_path"].isna().sum())

print("\nTrain image_path examples:")
print(train_df["image_path"].head())

In [7]:
# =======================================
# CHECK MISSING IMAGE PATHS BY DATASET
# =======================================

print("TRAIN missing paths by dataset:")
print(train_df[train_df["image_path"].isna()]["dataset"].value_counts())

print("\nVALIDATION missing paths by dataset:")
print(val_df[val_df["image_path"].isna()]["dataset"].value_counts())

print("\nTEST missing paths by dataset:")
print(test_df[test_df["image_path"].isna()]["dataset"].value_counts())

TRAIN missing paths by dataset:


NameError: name 'train_df' is not defined

In [10]:
# =======================================
# FIX ISIC IMAGE PATHS
# =======================================

import os
import glob

isic_image_dir = "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629/images"

# Get all ISIC image files
isic_image_files = glob.glob(
    os.path.join(isic_image_dir, "**", "*"),
    recursive=True
)

# Keep image files only
isic_image_files = [
    p for p in isic_image_files
    if p.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Total ISIC image files found:", len(isic_image_files))

# Create filename -> full path mapping
isic_image_map = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in isic_image_files
}

print("ISIC image mapping created:", len(isic_image_map))

# Match ISIC image paths using isic_id
def get_isic_image_path(row):
    image_id = str(row["isic_id"])

    return isic_image_map.get(image_id, None)


# Rebuild image_path for the complete ISIC dataframe
isic["image_path"] = isic.apply(get_isic_image_path, axis=1)

print("\n=======================================")
print("ISIC IMAGE PATH MATCHING")
print("=======================================")

print("Total ISIC records:", len(isic))
print("Valid image paths:", isic["image_path"].notna().sum())
print("Missing image paths:", isic["image_path"].isna().sum())

print("\nExample paths:")
print(isic[["isic_id", "image_path"]].head())

Total ISIC image files found: 81722
ISIC image mapping created: 81722

ISIC IMAGE PATH MATCHING
Total ISIC records: 81722
Valid image paths: 81722
Missing image paths: 0

Example paths:
        isic_id                                         image_path
0  ISIC_7559201  /kaggle/input/datasets/tomooinubushi/all-isic-...
1  ISIC_0485014  /kaggle/input/datasets/tomooinubushi/all-isic-...
2  ISIC_5257439  /kaggle/input/datasets/tomooinubushi/all-isic-...
3  ISIC_2989732  /kaggle/input/datasets/tomooinubushi/all-isic-...
4  ISIC_5638210  /kaggle/input/datasets/tomooinubushi/all-isic-...


In [ ]:
# ========================================
# CHECK NEW SCC / MCC DATASETS
# ========================================

import os

new_dataset_paths = {
    "ISIC_LABELLED":
        "/kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled",

    "MCC_DATASET":
        "/kaggle/input/datasets/quantumcoders05/mccdataset",

    "SKIN_CANCER_9":
        "/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic",

    "SKIN_DS":
        "/kaggle/input/datasets/ahmedxc4/skin-ds"
}

image_extensions = (".jpg", ".jpeg", ".png", ".webp")

print("=======================================")
print("NEW DATASET IMAGE SEARCH")
print("=======================================")

for dataset_name, dataset_path in new_dataset_paths.items():

    print("\n=======================================")
    print(dataset_name)
    print("=======================================")
    print("Path:", dataset_path)

    if not os.path.exists(dataset_path):
        print("❌ PATH NOT FOUND")
        continue

    total_images = 0

    for root, dirs, files in os.walk(dataset_path):

        image_files = [
            f for f in files
            if f.lower().endswith(image_extensions)
        ]

        if image_files:

            total_images += len(image_files)

            print("\nImages:", len(image_files))
            print("Folder:", root)
            print("Examples:", image_files[:5])

    print("\nTOTAL IMAGES:", total_images)

In [13]:
# ========================================
# COLLECT NEW SCC + MCC IMAGES
# ========================================

import os
import glob
import pandas as pd

# ----------------------------------------
# Dataset paths
# ----------------------------------------

scc_sources = [
    "/kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled/ISIC_Labelled/Squamous cell carcinoma",

    "/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test/squamous cell carcinoma",

    "/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Train/squamous cell carcinoma",

    "/kaggle/input/datasets/ahmedxc4/skin-ds/test/Squamous cell carcinoma",

    "/kaggle/input/datasets/ahmedxc4/skin-ds/val/Squamous cell carcinoma",

    "/kaggle/input/datasets/ahmedxc4/skin-ds/train/Squamous cell carcinoma"
]

mcc_sources = [
    "/kaggle/input/datasets/quantumcoders03/mcc-dataset/mcc image"
]

# ----------------------------------------
# Collect images
# ----------------------------------------

image_extensions = (".jpg", ".jpeg", ".png", ".webp")

new_scc_records = []
new_mcc_records = []

# SCC
for folder in scc_sources:

    if not os.path.exists(folder):
        print("⚠️ Missing:", folder)
        continue

    for root, dirs, files in os.walk(folder):

        for file in files:

            if file.lower().endswith(image_extensions):

                new_scc_records.append({
                    "image_name": file,
                    "image_path": os.path.join(root, file),
                    "class_name": "Squamous Cell Carcinoma",
                    "label": 2,
                    "dataset": "NEW_SCC"
                })


# MCC
for folder in mcc_sources:

    if not os.path.exists(folder):
        print("⚠️ Missing:", folder)
        continue

    for root, dirs, files in os.walk(folder):

        for file in files:

            if file.lower().endswith(image_extensions):

                new_mcc_records.append({
                    "image_name": file,
                    "image_path": os.path.join(root, file),
                    "class_name": "Merkel Cell Carcinoma",
                    "label": 3,
                    "dataset": "NEW_MCC"
                })


# ----------------------------------------
# Create DataFrames
# ----------------------------------------

new_scc_df = pd.DataFrame(new_scc_records)
new_mcc_df = pd.DataFrame(new_mcc_records)

print("\n=======================================")
print("NEW SCC / MCC DATA")
print("=======================================")

print("\nSCC images found:", len(new_scc_df))
print("MCC images found:", len(new_mcc_df))

print("\nSCC distribution:")
print(new_scc_df["class_name"].value_counts())

print("\nMCC distribution:")
print(new_mcc_df["class_name"].value_counts())

print("\nSCC examples:")
print(new_scc_df.head())

print("\nMCC examples:")
print(new_mcc_df.head())


NEW SCC / MCC DATA

SCC images found: 1453
MCC images found: 17

SCC distribution:
class_name
Squamous Cell Carcinoma    1453
Name: count, dtype: int64

MCC distribution:
class_name
Merkel Cell Carcinoma    17
Name: count, dtype: int64

SCC examples:
         image_name                                         image_path  \
0  ISIC_0028872.jpg  /kaggle/input/datasets/riyaelizashaju/isic-ski...   
1  ISIC_0058690.jpg  /kaggle/input/datasets/riyaelizashaju/isic-ski...   
2  ISIC_0055870.jpg  /kaggle/input/datasets/riyaelizashaju/isic-ski...   
3  ISIC_0072158.jpg  /kaggle/input/datasets/riyaelizashaju/isic-ski...   
4  ISIC_0072135.jpg  /kaggle/input/datasets/riyaelizashaju/isic-ski...   

                class_name  label  dataset  
0  Squamous Cell Carcinoma      2  NEW_SCC  
1  Squamous Cell Carcinoma      2  NEW_SCC  
2  Squamous Cell Carcinoma      2  NEW_SCC  
3  Squamous Cell Carcinoma      2  NEW_SCC  
4  Squamous Cell Carcinoma      2  NEW_SCC  

MCC examples:
                  

In [12]:
# =======================================
# CHECK MCC DATASET PATH
# =======================================

import os

mcc_root = "/kaggle/input/datasets/quantumcoders03/mcc-dataset"

print("MCC dataset exists:", os.path.exists(mcc_root))

print("\nFolders / files inside MCC dataset:")
for root, dirs, files in os.walk(mcc_root):
    print(root, "->", len(files), "files")

MCC dataset exists: True

Folders / files inside MCC dataset:
/kaggle/input/datasets/quantumcoders03/mcc-dataset -> 0 files
/kaggle/input/datasets/quantumcoders03/mcc-dataset/mcc image -> 17 files


In [16]:
# ========================================
# REBUILD 4-CLASS COMBINED DATASET
# ========================================

import pandas as pd
import os

print("=======================================")
print("REBUILDING 4-CLASS COMBINED DATASET")
print("=======================================")

# ----------------------------------------
# 1. ISIC DATA
# Melanoma + Basal Cell Carcinoma
# + Squamous Cell Carcinoma
# ----------------------------------------

isic_clean = isic_3class[
    ["image_path", "class_name", "label", "dataset"]
].copy()

isic_clean["image_name"] = isic_clean["image_path"].apply(
    lambda x: os.path.basename(str(x))
)

# ----------------------------------------
# 2. NEW SCC DATA
# ----------------------------------------

scc_clean = new_scc_df[
    ["image_name", "image_path", "class_name", "label", "dataset"]
].copy()

# ----------------------------------------
# 3. NEW MCC DATA
# ----------------------------------------

mcc_clean = new_mcc_df[
    ["image_name", "image_path", "class_name", "label", "dataset"]
].copy()

# ----------------------------------------
# 4. COMBINE ALL FOUR CLASSES
# ----------------------------------------

combined_df = pd.concat(
    [
        isic_clean,
        scc_clean,
        mcc_clean
    ],
    ignore_index=True
)

# ----------------------------------------
# 5. REMOVE MISSING IMAGE PATHS
# ----------------------------------------

combined_df = combined_df[
    combined_df["image_path"].notna()
].copy()

# ----------------------------------------
# 6. REMOVE DUPLICATE IMAGE PATHS
# ----------------------------------------

before = len(combined_df)

combined_df = combined_df.drop_duplicates(
    subset=["image_path"]
).reset_index(drop=True)

duplicates_removed = before - len(combined_df)

# ----------------------------------------
# 7. FINAL RESULTS
# ----------------------------------------

print("\n=======================================")
print("4-CLASS DATASET READY")
print("=======================================")

print("Total samples:", len(combined_df))
print("Duplicates removed:", duplicates_removed)

print("\nCLASS DISTRIBUTION")
print("---------------------------------------")
print(
    combined_df["class_name"].value_counts()
)

print("\nLABEL DISTRIBUTION")
print("---------------------------------------")
print(
    combined_df["label"].value_counts().sort_index()
)

print("\nIMAGE PATH CHECK")
print("---------------------------------------")
print(
    "Missing paths:",
    combined_df["image_path"].isna().sum()
)

print("\n=======================================")
print("FINAL CLASS COUNTS")
print("=======================================")

for class_name in [
    "Melanoma",
    "Basal Cell Carcinoma",
    "Squamous Cell Carcinoma",
    "Merkel Cell Carcinoma"
]:
    
    count = (
        combined_df["class_name"] == class_name
    ).sum()
    
    print(f"{class_name}: {count}")

print("\n=======================================")
print("LABEL MAPPING")
print("=======================================")

print("0 = Melanoma")
print("1 = Basal Cell Carcinoma")
print("2 = Squamous Cell Carcinoma")
print("3 = Merkel Cell Carcinoma")

print("\n=======================================")
print("COMBINED DATASET COMPLETE")
print("=======================================")

REBUILDING 4-CLASS COMBINED DATASET

4-CLASS DATASET READY
Total samples: 15112
Duplicates removed: 0

CLASS DISTRIBUTION
---------------------------------------
class_name
Melanoma                   7349
Basal Cell Carcinoma       4921
Squamous Cell Carcinoma    2825
Merkel Cell Carcinoma        17
Name: count, dtype: int64

LABEL DISTRIBUTION
---------------------------------------
label
0.0    7349
1.0    4921
2.0    2825
3.0      17
Name: count, dtype: int64

IMAGE PATH CHECK
---------------------------------------
Missing paths: 0

FINAL CLASS COUNTS
Melanoma: 7349
Basal Cell Carcinoma: 4921
Squamous Cell Carcinoma: 2825
Merkel Cell Carcinoma: 17

LABEL MAPPING
0 = Melanoma
1 = Basal Cell Carcinoma
2 = Squamous Cell Carcinoma
3 = Merkel Cell Carcinoma

COMBINED DATASET COMPLETE


In [5]:
# =======================================
# 4-CLASS RESNET18 - 1 EPOCH TEST
# =======================================

import time
import torch

print("=======================================")
print("STARTING 1-EPOCH TRAINING TEST")
print("=======================================")

num_epochs = 1

start_time = time.time()

model.train()

running_loss = 0.0
correct = 0
total = 0

for batch_idx, (images, labels) in enumerate(train_loader):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

    _, predicted = torch.max(outputs, 1)

    total += labels.size(0)
    correct += (predicted == labels).sum().item()

    if (batch_idx + 1) % 50 == 0:
        print(f"Batch [{batch_idx+1}/{len(train_loader)}]")

train_loss = running_loss / len(train_loader)
train_acc = 100 * correct / total

elapsed = time.time() - start_time

print("\n=======================================")
print("1-EPOCH TEST COMPLETE")
print("=======================================")
print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc:.2f}%")
print(f"Time       : {elapsed/60:.2f} minutes")
print("=======================================")

STARTING 1-EPOCH TRAINING TEST


NameError: name 'train_loader' is not defined

In [3]:
# =======================================
# SETUP 4-CLASS RESNET18 MODEL
# OFFLINE VERSION
# =======================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

print("=======================================")
print("SETTING UP 4-CLASS RESNET18")
print("=======================================")

# CPU
device = torch.device("cpu")
print("Device:", device)

# Create ResNet18 WITHOUT downloading weights
model = models.resnet18(weights=None)

# 4 cancer classes
model.fc = nn.Linear(model.fc.in_features, 4)

# Move model to CPU
model = model.to(device)

# Loss
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

print("Classes: 4")
print("Model: ResNet18")
print("Pretrained weights: OFFLINE / NONE")
print("Optimizer: Adam")
print("Learning rate: 1e-4")

print("=======================================")
print("MODEL SETUP COMPLETE")
print("=======================================")

SETTING UP 4-CLASS RESNET18
Device: cpu
Classes: 4
Model: ResNet18
Pretrained weights: OFFLINE / NONE
Optimizer: Adam
Learning rate: 1e-4
MODEL SETUP COMPLETE
